# LLM Fine-Tuning Deep Dive, Part 1 of 3: Data-Based Techniques

> **This is Part 1 of a three-notebook fine-tuning arc:**
>
> 1. **Part 1 (this notebook): Data-based techniques** -- what objective/data teaches the
>    behavior (continued pretraining, instruction tuning, preference alignment / DPO).
> 2. [Part 2: Parameter-based techniques + QLoRA & quantization](02-llm-finetuning-parameter-techniques.ipynb)
>    -- how many/which weights are updated (full fine-tuning, partial freezing, LoRA, QLoRA), plus
>    a real, runnable look at post-training quantization for deployment.
> 3. [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) -- head-to-head
>    evaluation of all six trained checkpoints, held-out perplexity, an ablation study, and the final
>    call on what Riverside House actually deploys.
>
> The two axes (data-based and parameter-based) are **independent choices**, which is exactly why
> they split cleanly into separate notebooks -- Part 1 answers "what should the model learn from,"
> Part 2 answers "how much of the model should move while it learns." Every checkpoint Part 1 trains
> is saved to `./checkpoints/` on disk, and Part 2 and Part 3 reload them from there rather than
> assuming this notebook's kernel is still running.

## The brief: Riverside House needs an in-house AI, not an API call

**Riverside House** is a small publishing firm. Everything under [`content/`](content/) is their
**unpublished, proprietary manuscript catalog** -- seven complete novels spanning sci-fi, fantasy,
mystery, historical fiction, cyberpunk, horror, and literary fiction, still under contract, still
unreleased. That's exactly why nobody at Riverside is allowed to paste chapters into a public chatbot
API: the moment draft manuscripts leave the building, the confidentiality clause is broken. Whatever
model they use has to be trained and run **in-house**, on hardware they already own -- a laptop, not a
GPU cluster, and no data ever leaves it.

Riverside's ask has two parts:

1. **An editing assistant** for the ghostwriters and continuity editors -- prompt it with a scene and
   get back a continuation that actually remembers who Aria Voss is and what the Meridian's Promise
   is, follows a direct instruction instead of rambling forever, and reads the way their editors
   _actually_ prefer.
2. **A knowledge base for the rest of the company** -- marketing, licensing, and new hires who need
   answers like "who are the six founding families in the mystery novel?" without reading 197 chapters
   or, worse, guessing.

Today, every one of those people either re-reads old chapters by hand or asks a colleague. That's the
gap this notebook closes -- and by the end we have to pick **one model to actually deploy**, backed by
more than "it read fine to me."

**Goal:** fine-tune a small-but-capable base model (`gpt2-medium`, ~355M parameters) on Riverside's
proprietary catalog -- **seven complete novels** (~619,000 words / ~3.0 MB total, entirely inside
[`content/`](content/)) -- so it learns their characters, invented terminology, and prose style across
genres, demonstrating **every major axis of fine-tuning** along the way:

| Step | Concept                            | Riverside's question                                                                         | Notebook |
| ---- | ----------------------------------- | ---------------------------------------------------------------------------------------------| -------- |
| 1    | Continued pretraining              | Does it even know our characters and world exist?                                            | Part 1 (this one) |
| 2    | Instruction tuning (SFT)           | Does it follow a "continue this scene" / "answer this question" request instead of rambling? | Part 1 (this one) |
| 3    | Preference alignment (DPO)         | Does it write and answer the way our editors actually prefer?                                 | Part 1 (this one) |
| 4    | Full fine-tuning                   | Best quality -- but what does it cost on a laptop?                                           | Part 2 |
| 5    | Partial freezing                   | A cheaper middle ground -- how much quality do we give up?                                   | Part 2 |
| 6    | LoRA + QLoRA                       | The cheapest option -- is it good enough to ship? What if we quantize too?                   | Part 2 |
| 7    | Ablation study                     | What breaks if the deadline forces us to skip a stage?                                       | Part 3 |
| 8    | Head-to-head + held-out perplexity | Which model do we actually deploy in-house?                                                  | Part 3 |

| Axis                | Question it answers                         | Techniques covered here                                                                           |
| ------------------- | ------------------------------------------- | ------------------------------------------------------------------------------------------------- |
| **Data-based**      | _What objective/data teaches the behavior?_ | Non-instructional (continued pretraining), Instructional (supervised), Preference alignment (DPO) |
| **Parameter-based** | _How many/which weights are updated?_       | Full fine-tuning, Partial (layer freezing), Parameter-efficient (LoRA), QLoRA (Part 2)             |

## Corpus (Riverside House's proprietary manuscripts)

- **Location:** [`content/`](content/) -- **7 original, unpublished novels** across diverse genres
  (sci-fi, fantasy, mystery, historical, cyberpunk, horror, literary), totaling **197 chapters**
  (~619,000 words / ~3.0 MB). This directory _is_ Riverside's confidential catalog for the purposes
  of this notebook. See [`content/README.md`](content/README.md) for the full breakdown and
  individual synopses.
- **Genres available:**
  - Sci-fi: _The Weight of Distant Light_ (generation ship, 40 chapters)
  - Fantasy: _The Tidebound Accord_ (epic quest, 33 chapters)
  - Mystery: _The Cartographer's Cipher_ (noir detective, 21 chapters)
  - Historical: _The Silk Merchant's Daughter_ (Tang Dynasty, 23 chapters)
  - Cyberpunk: _Neural Drift_ (memory broker conspiracy, 24 chapters)
  - Horror: _The Hollow Beneath_ (gothic/cosmic, 28 chapters)
  - Literary: _The Weight of Tides_ (marine biology first contact, 28 chapters)
- **Scale note:** Training cells sample a subset (`max_chapters` per genre) by default for fast CPU
  demos. Pass larger limits or `genres=None` to train on the full corpus.

## Setup

Run `setup.ps1` once to create a `.venv` and register the `llm-tuning` Jupyter kernel, then select
that kernel for this notebook.


## Table of Contents (Part 1 of 3)

1. [The Brief: Riverside House Needs an In-House AI](#the-brief-riverside-house-needs-an-in-house-ai-not-an-api-call)
   - [Corpus](#corpus-riverside-houses-proprietary-manuscripts)
   - [Setup](#setup)
2. [Why Fine-Tuning? The Three-Gap Problem](#why-fine-tuning-the-three-gap-problem)
3. [Baseline: What Does the Un-Tuned Model Know?](#baseline-what-does-the-un-tuned-model-know)
   - [Setting Up: Which Model, and Why](#setting-up-which-model-and-why)
   - [Choosing a Device](#choosing-a-device-gpu-if-available-cpu-otherwise)
   - [Loading the Tokenizer](#loading-the-tokenizer)
   - [Seeing the Vocabulary in Action](#seeing-the-vocabulary-in-action)
   - [Loading the Base Model](#loading-the-base-model)
   - [A Fixed Test Prompt for Before/After Comparisons](#a-fixed-test-prompt-for-beforeafter-comparisons)
   - [A Reusable `generate()` Helper](#a-reusable-generate-helper)
4. [Test Prompts for Validating Fine-Tuning](#test-prompts-for-validating-fine-tuning)
   - [Understanding One Training Step: A Concrete Example](#understanding-one-training-step-a-concrete-example)
5. [The Fine-Tuning Journey: A Problem-Solution Narrative](#the-fine-tuning-journey-a-problem-solution-narrative)
6. [Concept 1: Continued Pretraining](#concept-1-data-based-non-instructional-fine-tuning-continued-pretraining)
   - [Common Pitfalls: Continued Pretraining](#common-pitfalls-continued-pretraining)
7. [Concept 2: Instruction Tuning (SFT)](#concept-2-data-based-instructional-supervised-fine-tuning)
   - [Common Pitfalls: Instruction Tuning](#common-pitfalls-instruction-tuning)
8. [Concept 3: Preference Alignment (DPO)](#concept-3-data-based-preference-alignment-rlhf--dpo)
   - [DPO vs. PPO Comparison](#dpo-vs-ppo-two-ways-to-optimize-the-same-preference-data)
   - [Common Pitfalls: DPO](#common-pitfalls-dpo-preference-alignment)

> Links jump to the matching heading below. If a link doesn't scroll correctly in your Jupyter
> viewer, use `Ctrl+F` / the notebook outline panel with the section title instead -- the numbered
> list above still tells you the order and grouping of everything in this notebook.

**Continues in:**
- [Part 2: Parameter-based techniques + QLoRA & quantization](02-llm-finetuning-parameter-techniques.ipynb)
  -- full fine-tuning, partial freezing, LoRA, QLoRA, and a real post-training quantization demo.
- [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) -- head-to-head
  evaluation, held-out perplexity, ablation study, and the final deployment decision.

---


In [ ]:
from pathlib import Path

# Resolve the notebook's own directory (content/ lives next to this notebook). VS Code's Jupyter
# kernels run with cwd = workspace root, not the notebook's folder, so __vsc_ipynb_file__ (which
# VS Code injects) is the reliable way to find it; __file__ covers plain .py execution.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():
    # Fallback for kernels whose cwd is the repo root instead of this notebook's own folder
    _fallback = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if _fallback.exists():
        CONTENT_DIR = _fallback

print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel directory mappings (keys are shorthand aliases, values are actual directory names)
NOVELS = {
    "scifi":      "the-weight-of-distant-light",   # 40 chapters
    "fantasy":    "the-tidebound-accord",           # 33 chapters
    "mystery":    "the-cartographers-cipher",       # 21 chapters
    "historical": "the-silk-merchants-daughter",    # 23 chapters
    "cyberpunk":  "neural-drift",                   # 24 chapters
    "horror":     "the-hollow-beneath",             # 28 chapters
    "literary":   "the-weight-of-tides",            # 28 chapters
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from the multi-novel corpus for quick CPU demos.

    Args:
        novels: List of novel aliases to load (e.g., ["scifi", "fantasy"]), or None to
                load all. Available aliases: "scifi", "fantasy", "mystery", "historical",
                "cyberpunk", "horror", "literary" (mapped to directory names).
        max_chapters: Max chapters to load per novel (keeps CPU training fast).
        min_len: Skip paragraphs shorter than this many characters.

    Returns:
        List of paragraph strings from all requested novels.
    """
    if novels is None:
        novels = list(NOVELS.keys())  # load all by default

    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            print(f"Warning: unknown novel alias '{alias}', skipping")
            continue

        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            print(f"Warning: directory {novel_path} not found, skipping")
            continue

        chapter_files = sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)

    return paragraphs


# Sample from 4 genres to show multi-genre paragraph diversity
sample_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery", "horror"], max_chapters=2
)
print(
    f"Loaded {len(sample_paragraphs)} sample paragraphs from 4 novels. First one:\n"
)
print(sample_paragraphs[20][:421], "...")

## Why Fine-Tuning? The Three-Gap Problem

A pretrained LLM (like GPT-2, LLaMA, or Mistral) has learned language from billions of tokens of web
text, books, and code. This gives it strong **general fluency** and **broad world knowledge**. That's
exactly why `gpt2-medium` is a reasonable starting point for Riverside House's assistant -- it can
already write fluent English. But for Riverside's specific job, it has three concrete gaps:

### Gap 1: Domain Knowledge Gap

**Problem:** The model has never seen Riverside's specific vocabulary, characters, facts, or house
style.

**Example with our corpus:**

- **An editor asks:** `"Who is Aria Voss?"`
- **Base model:** `"Aria Voss is a... [makes up something generic or says 'I don't know']"`
- **After fine-tuning:** `"Aria Voss is the Hold systems technician aboard the Meridian's Promise 
generation ship..."`

**Solution:** Continued pretraining on Riverside's catalog.

---

### Gap 2: Behavior Gap

**Problem:** A raw pretrained model just continues text. It doesn't know how to follow instructions,
answer questions directly, or stop when it should -- which is a problem the moment a ghostwriter wants
to _ask_ something instead of just seeding a paragraph.

**Example:**

- **An editor asks:** `"List the five tides in the Tidebound Accord."`
- **After domain pretraining:** `"List the five tides in the Tidebound Accord. This question has 
puzzled scholars for millennia. Some say there are actually six tides, while others..."` (rambles
  forever)
- **After instruction tuning:** `"The five tides are: water, wind, stone, flame, and void."`

**Solution:** Instruction tuning (supervised fine-tuning) on (prompt, completion) pairs.

---

### Gap 3: Preference Gap

**Problem:** Even an instruction-following model may produce outputs that are technically correct but
not what Riverside's editors actually prefer (too verbose, wrong tone, unhelpful focus) -- and "reads
worse than a human editor would tolerate" is exactly the kind of gap that kills adoption of an internal
tool, even when it "technically works."

**Example:**

- **An editor asks:** `"Explain quantum entanglement simply."`
- **After instruction tuning:** `"Quantum entanglement is a phenomenon in quantum mechanics wherein 
the quantum states of two or more particles become interdependent such that the state of one cannot 
be fully described without reference to the others, even when separated by large distances..."` (10
  more paragraphs of jargon)
- **After preference alignment:** `"Quantum entanglement means two particles become connected so 
measuring one instantly affects the other, even across vast distances."`

**Solution:** Preference alignment (RLHF or DPO) using human preference data.

---

### One Journey, Three Gaps to Close

Which stage to start from -- and how far down this pipeline to go -- depends on which gap is actually
blocking Riverside's assistant today:

```
Pretrained base ---> Continued pretraining ---> Instruction tuning ---> Preference alignment ---> Production model
    (Gap 0)               (Closes Gap 1)              (Closes Gap 2)            (Closes Gap 3)
```

At each stage, you also choose **how many parameters to update** (full fine-tuning vs. freezing vs.
LoRA) -- which, for Riverside House, is really a budget question: they have a laptop, not a GPU
cluster. We'll explore that trade-off after demonstrating the data-based journey.

---

## Baseline: What Does the Un-Tuned Model Know?

Before fine-tuning, let's see what `gpt2-medium` (pretrained on generic web text) produces when
prompted with a scenario from Riverside's sci-fi novel. Since it has never seen this story, expect a
fluent but generic, off-world continuation -- this is the starting point Riverside's editors are stuck
with today.


### Setting Up: Which Model, and Why

Every technique in this notebook needs a base model to fine-tune, so the first thing to pin down is
*which* pretrained checkpoint we're starting from -- everything downstream (the tokenizer, the
device, every training loop later in the notebook) depends on this one choice.

`MODEL_NAME` is set once, here, and reused everywhere else in the notebook
(`AutoTokenizer.from_pretrained(MODEL_NAME)`, `AutoModelForCausalLM.from_pretrained(MODEL_NAME)`, and
every training cell that loads a fresh copy to fine-tune), so swapping to a bigger or different base
model later only ever means changing this one string.

We pick `gpt2-medium` (~355M params) -- real, pretrained weights, not a toy -- because it's still
small enough to fine-tune on a CPU in a few minutes per stage, which is exactly Riverside's
one-laptop constraint.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2-medium"  # ~355M params, real pretrained weights, still CPU-trainable but far better suited to actually absorbing a domain corpus than distilgpt2

### Choosing a Device: GPU if Available, CPU Otherwise

`device` tells PyTorch where tensors and model weights should physically live. We need this because
every tensor operation in this notebook (forward pass, backward pass, `.generate()`) has to run on
the same device as the weights, or PyTorch raises a device-mismatch error.

`torch.cuda.is_available()` checks for a usable NVIDIA GPU + CUDA driver; if none is found, we fall
back to `"cpu"` so the notebook still runs end-to-end (just slower) on a laptop with no dedicated
GPU -- Riverside's actual situation.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### Loading the Tokenizer

`gpt2-medium` never sees a single letter of Riverside's manuscripts directly -- it's a stack of
matrix multiplications, and matrices only take numbers. That same constraint applied long before this
notebook existed: to pretrain the model in the first place, billions of tokens of web text had to be
converted into integer IDs using a fixed vocabulary built with byte-pair encoding (BPE), and every
weight in `gpt2-medium`'s embedding matrix and output head was learned against that exact, frozen set
of IDs. Nothing about those weights has changed since -- which is exactly why we can't swap in just
any tokenizer here. Token ID 464 only means whatever `gpt2-medium`'s vocabulary says it means; a
different model's tokenizer would map that same integer to a different, or meaningless, piece of
text -- not a crash, just quietly wrong output. That's why Hugging Face ships every checkpoint paired
with its own tokenizer, and why `AutoTokenizer.from_pretrained(MODEL_NAME)` below is keyed off the
exact same `MODEL_NAME` string used to load the model itself.

One gap remains once that tokenizer is loaded. GPT-2 was pretrained on one continuous stream of
concatenated documents, chopped into fixed-length blocks -- every training example was already
exactly the right length, so padding shorter sequences never came up, and no pad token was ever
defined. This notebook's fine-tuning runs are different: Riverside's chapters get split into
individual paragraphs of wildly different lengths, batched together for speed, and a batched tensor
operation needs every sequence in the batch to share one shape. So right after loading the tokenizer,
we hand it a pad token to borrow -- reusing `eos_token`, the standard convention for GPT-family
models, since it fills the gap without adding a new row to the embedding matrix.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### Seeing the Vocabulary in Action

Before moving on, it's worth actually looking at what `tokenizer` just handed us instead of taking it
on faith. GPT-2's vocabulary is a byte-level BPE (byte-pair encoding) table, and its merges were
learned by counting which byte sequences showed up most often in a huge corpus of ordinary web text.
Crucially, GPT-2's tokenizer treats a leading space as *part of* the token, not as a separate
character -- so `"signal"` and `" signal"` (with a space in front) are two completely unrelated byte
sequences as far as BPE is concerned, even though they look like "the same word" to a human. Whichever
sequence occurred more often during training earned its own single merged token; the less common one,
if it never came up often enough, stayed split into smaller pieces.

That produces a small surprise below: `signal`, `stared`, and `distant` each split into two pieces
when tokenized alone -- as if they were the very first word of a document, with nothing before them
-- but collapse into a *single* token the instant they get their leading space back (`Ġsignal`,
`Ġstared`, `Ġdistant`), since `" signal"`-style sequences are everywhere in ordinary prose while the
bare, space-less form is comparatively rare. `Aria` splits into two tokens either way, since even its
common, space-led form (`" Aria"`) wasn't frequent enough in GPT-2's training data to earn a dedicated
token -- it's a name, not an everyday word. (`Ġ` is just how this tokenizer prints "a space came right
before this" when it decodes a token back to a readable string.)

In [ ]:
# One example each of a noun, proper noun, verb, and adjective -- all pulled from Riverside's own
# sci-fi opening line, so these are words this notebook already leans on elsewhere.
example_words = {
    "noun": "signal",
    "proper noun": "Aria",
    "verb": "stared",
    "adjective": "distant",
}

for part_of_speech, word in example_words.items():
    ids_alone = tokenizer.encode(word)
    ids_mid_sentence = tokenizer.encode(" " + word)
    print(f"{part_of_speech.upper()}: {word!r}")
    print(
        f"  as the first word of a text  : ids={ids_alone}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_alone)}"
    )
    print(
        f"  mid-sentence (' {word}')".ljust(31)
        + f": ids={ids_mid_sentence}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_mid_sentence)}"
    )
    print()

print(
    "'\u0120' at the start of a token marks a leading space -- it's why the same word can tokenize "
    "differently depending on where it appears in a sentence."
)

### Loading the Base Model

`base_model` is the actual pretrained neural network -- 355M real weights downloaded from the
Hugging Face hub, moved onto whichever device we resolved above via `.to(device)`. This untouched
checkpoint is the "before" every fine-tuning technique in this notebook is compared against.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

### A Fixed Test Prompt for Before/After Comparisons

`PROMPT` is the one fixed test sentence reused throughout the notebook so "before" vs. "after"
fine-tuning comparisons are always apples-to-apples. It's pulled straight from the sci-fi corpus so
a model that has actually absorbed the catalog has a real chance of continuing it in-world.

In [ ]:
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus

### A Reusable `generate()` Helper

`generate()` is a small wrapper around HuggingFace's `model.generate()` that every later section of
this notebook reuses to compare checkpoints. We need our own wrapper (instead of calling
`model.generate()` directly everywhere) because `model.generate()` returns the prompt tokens *and*
the new tokens concatenated together in one tensor -- there's no built-in "just give me the
continuation" option.

Here's what that actually looks like, using this notebook's own `PROMPT` (which tokenizes to 15
tokens) and a real generation call asking for 15 new tokens:

```
tokenizer(PROMPT)["input_ids"]          -> 15 tokens          (prompt_len = 15)
model.generate(..., max_new_tokens=15)  -> shape (1, 30)      (15 prompt + 15 new, concatenated)
```

Decoding the full, unsliced tensor prints the prompt right back at you, glued to the front of the
actual answer:

> "Aria Voss stared at the signal counting itself out in prime numbers and began to ponder the
> question, what was it that she had to do?"

`prompt_len` exists so `out[0][prompt_len:]` can drop those first 15 tokens -- without it, every
`print(generate(...))` in this notebook would repeat the whole prompt before showing anything new.

There's a second, smaller wrinkle once only the new tokens are decoded: the same real call above
decodes to `" began to ponder the question, what was it that she had to do?"` -- note the stray
leading space. The first generated token is almost always a space-led token (recall the `Ġ` prefix
from the tokenizer-vocabulary demo earlier), so the raw decode nearly always starts with one.
`.strip()` removes it, along with any trailing whitespace/newlines the model happens to generate
near the end.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    """Generate a text continuation for *prompt*.

    Returns **only the newly generated tokens** (prompt is stripped), so every
    print(generate(...)) call in this notebook shows the model's actual output
    without echoing the input back.

    Parameters
    ----------
    model : PreTrainedModel or PeftModel
        Any HuggingFace causal-LM model (base GPT-2, LoRA adapter, DPO policy …)
    prompt : str
        The input text passed to the model.
    max_new_tokens : int
        Hard cap on how many new tokens to generate after the prompt ends.
        The model can stop earlier if it samples the EOS token.

    Notes
    -----
    A real example from this notebook's own `PROMPT` (15 tokens) makes both
    lines concrete. Asking for `max_new_tokens=15` returns `out` with shape
    `(1, 30)` -- the 15 prompt tokens plus 15 new ones, concatenated. Decoding
    all 30 without slicing prints the prompt right back before the answer:

        'Aria Voss stared at the signal counting itself out in prime numbers
         and began to ponder the question, what was it that she had to do?'

    `out[0][prompt_len:]` (`prompt_len = 15` here) drops the first 15 tokens so
    only the new continuation gets decoded. But decoding *just* those 15 new
    tokens gives:

        ' began to ponder the question, what was it that she had to do?'

    -- note the stray leading space: the first generated token is almost
    always a space-led token (the `Ġ` prefix from the tokenizer-vocabulary
    demo above), so the raw decode nearly always starts with one. `.strip()`
    removes it, along with any trailing whitespace/newlines near the end.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device) # use the same tokenizer to tokenize the prompt and convert it to tensor
    prompt_len = inputs["input_ids"].shape[1]          # track where the prompt ends
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,                            # stochastic → varied output
            top_p=0.9,                                 # nucleus sampling: top 90% mass
            temperature=0.8,                           # soften distribution slightly
            pad_token_id=tokenizer.pad_token_id,
        )
    # out[0] shape: (prompt_len + new_tokens,)
    # Slice from prompt_len onward to get ONLY the model's continuation
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return completion if completion else "[model stopped immediately — sampled EOS as first token]"


print(f"Completion: {generate(base_model, PROMPT)}")

print("=== Baseline (no fine-tuning) — model continuation only ===")
print(f"Prompt    : {PROMPT}")

### Code Walkthrough: Setup Cell

**What just ran — three building blocks used throughout this entire notebook:**

---

**1. `tokenizer.pad_token = tokenizer.eos_token`**

GPT-2 was pretrained on sequences of a fixed length with no padding token in its vocabulary. When the `Trainer` batches examples of different lengths it needs a pad token to fill the shorter sequences. Setting it to `eos_token` (ID 50256) is the standard convention — it tells the tokenizer "treat end-of-sequence as padding." The matching `labels=-100` mask reappears in `tokenize_causal()`, introduced in the Concept 1 section below where the notebook first actually needs to tokenize a training batch.

---

**2. `AutoModelForCausalLM.from_pretrained(MODEL_NAME)`**

This loads the full `gpt2-medium` checkpoint: 355 million parameters, 24 transformer blocks, hidden size 1024. The `.to(device)` call moves all weight tensors to CPU (or GPU if available). The `AutoModel` family is generic — swap `MODEL_NAME` for `"meta-llama/Llama-3.1-8B"` and the rest of the loading code adapts automatically.

---

**3. `generate(model, prompt, max_new_tokens=60)` — decoding strategy**

A thin wrapper around HuggingFace's `model.generate()` with three key choices:

| Parameter         | Value               | Effect                                                   |
| ----------------- | ------------------- | -------------------------------------------------------- |
| `do_sample=True`  | Stochastic decoding | Avoids repetitive, deterministic greedy output           |
| `top_p=0.9`       | Nucleus sampling    | Considers only tokens whose cumulative probability ≥ 90% |
| `temperature=0.8` | Slight smoothing    | Reduces "safe" word dominance without full randomness    |

The function returns **only the newly generated tokens** (after slicing off the prompt), so every `print(generate(...))` call shows the model's actual continuation — not the prompt echoed back at you.

> **PyTorch shape note:** `out = model.generate(...)` returns a tensor of shape `(batch=1, total_len)` where `total_len = prompt_len + max_new_tokens`. We slice `out[0][prompt_len:]` to get just the new tokens.

## Test Prompts for Validating Fine-Tuning

Use these prompts to test whether fine-tuning successfully absorbed domain-specific knowledge from
the seven-novel corpus. A well-tuned model should recognize characters, settings, and continue
narratives in the appropriate style. The baseline model (pretrained only) should produce generic,
off-topic continuations.

### What "success" actually looks like

"Generic" and "on-corpus" are easy to say but vague until you see them side by side. Take the first
sci-fi prompt above, `"Aria Voss checked the Meridian's Promise status panel and"`:

Prompt: `"Aria Voss checked the Meridian's Promise status panel and"`

**Baseline (no fine-tuning) -- what you'll actually see below:**

> A fluent but unrelated continuation -- often a different spaceship, a different job for "Aria," or
> generic technobabble. GPT-2's pretraining corpus has plenty of generic sci-fi, so it always produces
> *something* readable; it just isn't Riverside's story.

**Expected after continued pretraining on the corpus:**

> A continuation that stays consistent with the real chapter-001 setup: Aria as an engineer who
> monitors the ship "the way other people listened to weather," references to the Under-Hold (the
> ship's off-the-books maintenance underlayer), node seventeen (where she first notices the anomaly),
> or the Lantern (the alien signal the first novel revolves around) -- and a slow-burn, introspective
> tone rather than action-movie beats.

The mystery prompt makes the same point with a different failure mode. Take
`"The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—"`:

**Baseline:** invents a generic, historically-flavored sentence about "founding families" in the
abstract -- it has no idea these are six specific named characters from a land-fraud conspiracy.

**Expected after continued pretraining:** a continuation that stays anchored to the real plot --
the 1879 land fraud in Ashmont Bay, the falsified surveys, Mordecai's murder when he tried to
blackmail the others -- the kind of specific, checkable detail a generic pretrained model has no way
to produce because it was never shown Riverside's unpublished manuscript.

That's the bar every fine-tuning technique in this notebook is measured against: not "does the text
sound plausible" (the baseline already clears that bar), but "does it use the actual names, places,
and plot facts from the corpus." The baseline run right below makes the *before* half of that
comparison concrete; the Ablation Study near the end of this notebook (Experiment 1) revisits this
exact prompt with a real trained-vs-untrained side-by-side.

### Character & Setting Recognition Tests

**Sci-Fi (The Weight of Distant Light):**

- `"Aria Voss checked the Meridian's Promise status panel and"`
- `"The Keeper's consciousness flickered through node seventeen as"`
- `"In the Under-Hold, Nyla Kade whispered about the prime number signal from"`

**Fantasy (The Tidebound Accord):**

- `"Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as"`
- `"The ancient pillars rose from the Abyssal Rift while Davin Shale"`
- `"The Hollow King's followers, called the Hollowed,"`

**Mystery (The Cartographer's Cipher):**

- `"Elena Voss studied the 1879 survey map and realized the Ashmont Trust"`
- `"Detective Chen examined Adelaide Thorne's body and found the message: 'the foundation must hold'"`
- `"The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—"`

**Historical (The Silk Merchant's Daughter):**

- `"Wei Lian's jade phoenix pendant caught the morning light in Chang'an as"`
- `"Zhang Ming, the jinshi degree holder, wrote in his letter"`
- `"In the Eastern Market, the Wei family silk compound"`

**Cyberpunk (Neural Drift):**

- `"Kai Chen adjusted the neurorig and prepared to extract the memory backup from"`
- `"In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift"`
- `"Victor Tang's consciousness transfer protocol failed when"`

**Horror (The Hollow Beneath):**

- `"Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor"`
- `"The hollow beneath the house breathed, and the entity in the limestone caves"`
- `"Margot found Thaddeus Blackwood's journal warning: never descend past the second chamber"`

**Literary (The Weight of Tides):**

- `"Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about"`
- `"In Willowport, the underwater object near Whitehead Island caused"`
- `"The lobster traps came up bent, and the water temperature dropped fifteen degrees when"`

### Genre Style Continuation Tests

**Sci-Fi narrative momentum:**

- `"Two hundred and fourteen years after the Meridian's Promise left Earth,"`

**Fantasy elemental magic:**

- `"The tide-weavers gathered at Deepwater Crossing as the fifth tide, the void tide,"`

**Mystery noir atmosphere:**

- `"The rain-slicked streets of Ashmont Bay hid secrets from 1879, and Elena Voss"`

**Historical detail & restraint:**

- `"The silk road brought more than trade goods to Tang Dynasty Chang'an—it brought"`

**Cyberpunk tech-noir:**

- `"Memory extraction left traces, neural signatures that couldn't be scrubbed, and Kai Chen"`

**Gothic horror tension:**

- `"The house chose its inhabitants through grief, calling them when they were most vulnerable, and"`

**Literary introspection:**

- `"The ocean held its own memory, deeper and older than human documentation, and Claire"`

### Cross-Novel Vocabulary Tests

These should work across multiple genres if fine-tuning absorbed the corpus style:

- `"The weight of distant"` (tests sci-fi novel phrase bleed)
- `"The tidebound"` (tests fantasy terminology)
- `"permanent removal"` (tests mystery euphemism)
- `"steel in your spine, even if you must hide it beneath"` (tests historical voice)
- `"neural backup"` (tests cyberpunk jargon)
- `"the hollow"` (tests horror atmospheric language)
- `"The water's wrong"` (tests literary marine biology voice)


In [ ]:
# Automated test runner: compare baseline vs fine-tuned on corpus-specific prompts
TEST_PROMPTS = {
    # Sci-fi — The Weight of Distant Light
    "scifi_character": "Aria Voss checked the Meridian's Promise status panel and",
    "scifi_keeper": "The Keeper's consciousness flickered through node seventeen as",
    # Fantasy — The Tidebound Accord
    "fantasy_magic": "Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as",
    "fantasy_hollow_king": "The Hollow King's followers, called the Hollowed, began to gather when",
    # Mystery — The Cartographer's Cipher
    "mystery_conspiracy": "The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—",
    "mystery_elena": "Elena Voss studied the 1879 survey map and realized the Ashmont Trust",
    # Historical — The Silk Merchant's Daughter
    "historical_setting": "Wei Lian's jade phoenix pendant caught the morning light in Chang'an as",
    "historical_silk_road": "The delegation crossed the Taklamakan desert and Wei Lian noted in her ledger",
    # Cyberpunk — Neural Drift
    "cyberpunk_tech": "Kai Chen adjusted the neurorig and prepared to extract the memory backup from",
    "cyberpunk_project": "In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift",
    # Horror — The Hollow Beneath
    "horror_atmosphere": "Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor",
    "horror_chambers": "The tenth chamber of the Hollow pulsed with a light that had no source, and Eleanor",
    # Literary — The Weight of Tides
    "literary_marine": "Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about",
    "literary_contact": "The Observer surfaced near Whitehead Island and Claire understood for the first time that",
}


def test_corpus_knowledge(model, test_prompts=TEST_PROMPTS, max_new_tokens=50):
    """Run all test prompts and return results dict for comparison."""
    results = {}
    for key, prompt in test_prompts.items():
        results[key] = generate(model, prompt, max_new_tokens=max_new_tokens)
    return results


# Run baseline tests (will show generic, off-corpus continuations)
print(
    "=== BASELINE MODEL (no fine-tuning) - should produce generic continuations ===\n"
)
baseline_results = test_corpus_knowledge(base_model)
for key, output in baseline_results.items():
    print(f"[{key}]")
    print(output[:200] + "...\n")

Notice the output has no awareness of Aria Voss (from _The Weight of Distant Light_), the _Meridian's
Promise_, the Lantern, or any of the other characters/worlds across the seven novels -- it is fluent
English but a generic, unrelated continuation. This is exactly the gap fine-tuning closes.

Per the "What 'success' actually looks like" example above, a model that had genuinely absorbed this
corpus would instead keep Aria aboard the Meridian's Promise, in her actual role, referencing the
Under-Hold or the Lantern instead of inventing an unrelated ship and crew. This notebook doesn't
re-run `test_corpus_knowledge()` on a fine-tuned checkpoint (fine-tuning a fresh model per test prompt
would multiply the compute cost of every section below), but the Ablation Study near the end trains
and compares checkpoints on a closely related Aria Voss / Meridian's Promise prompt, so you can see
the real before/after side by side.



### Understanding One Training Step: A Concrete Example

Before we start training, let's walk through **exactly what happens** during a single training step of
continued pretraining, using a real paragraph and the real tokenizer -- the code cells further down
run the actual numbers below through `gpt2-medium` instead of making them up.

**Input:** A paragraph from our sci-fi corpus:

> "Aria Voss stared at the signal counting itself out in prime numbers and felt the weight of two
> centuries press against her ribs."

**Step 1: Tokenization**

The tokenizer converts text -> integer IDs, then pads (or truncates) to `max_length=128` so every
example in a batch is the same shape. Concretely, `tokenizer(...)` hands back **two** parallel
128-length arrays for our one-sentence example: `input_ids` (the real token integers, followed by
padding filled with `eos_token`) and `attention_mask` (`1` for every real token, `0` for every padding
slot). That `attention_mask` isn't just bookkeeping -- it's exactly what Step 2 below reads to decide
which of those 128 positions get `-100` in `labels`.

**Step 2: Create Labels for Causal LM -- this is the mask layout**

Causal-LM training is usually summarized in one line: *"the label at every position is just the input
shifted one to the left."* As a mental model, that's the right idea -- position `i`'s job is to
predict whatever token sits at position `i+1`. Taken literally, though, it makes it sound like we
build a whole second array, offset by one:

```
Input IDs: [t0,  t1,  t2,  t3,  ...]
Labels:    [t1,  t2,  t3,  t4,  ...]   # a separate, shifted array?
```

**That's not what the code further down actually builds.** It does this instead:
`labels = input_ids.clone()`, then every padding position gets overwritten with `-100`. `labels`
starts out **identical** to `input_ids` -- same tokens, same order, nothing rearranged. If tokenizing
our example produced `[Aria, Voss, stared, ...]`, then (ignoring padding) `labels` is also
`[Aria, Voss, stared, ...]` -- not the offset array shown above.

**So where does the "shift" actually happen, if not in `labels`?** In how the loss lines the two
*identical* sequences up afterward. `base_model(input_ids=..., labels=...)`, and the explicit
`shift_logits`/`shift_labels` lines you'll see further down, pair **the logits produced after
reading tokens `0..i`** with **the label sitting one position later, at `i+1`**:

```
Model has read:  [Aria]              -> its logit here is a guess at the NEXT token -> graded against label[1] = Voss
Model has read:  [Aria, Voss]        -> its logit here is a guess at the NEXT token -> graded against label[2] = stared
```

Every position's output is a prediction about what comes *after* it, so it only makes sense to grade
that output against the label one slot ahead -- `logits[:, :-1, :]` compared to `labels[:, 1:]`. GPT-2's
own `forward()` performs this exact `[:-1]` / `[1:]` alignment automatically the moment you pass it
`labels=...`; the `shift_logits`/`shift_labels` code below just redoes that same alignment explicitly,
so we can print the per-token loss instead of only the averaged total the model returns. The "shifted
input" from the opening one-liner is real -- it just happens as an indexing trick at loss time, on two
copies of the same array, rather than as a second array we build ourselves.

**The only masking that happens to `labels` itself is on padding:** any position past the real text is
set to `-100`, so the loss ignores it there. Every real token position stays active and gets compared
the way described above. The four-panel figure right after the real tokenization below walks through
this exact split step by step, using our example paragraph's actual tokens -- contrast it with the
instruction-tuning mask layout later in the notebook, where the **prompt** is also masked out, not just
the padding.

**Step 3: Forward Pass (Model Prediction)**

The model processes the **entire 128-token input in one shot** and outputs logits (unnormalized
scores) for every one of those 128 positions **simultaneously**. This is the actual mechanical reason
Step 2's "every position predicts what comes next" works at all: unlike an RNN, which reads tokens
one at a time, a transformer produces all 128 next-token guesses in a single forward pass --
`base_model(input_ids=...)` is one function call, not a loop over positions.

```
Logits shape: (batch_size=1, seq_len=128, vocab_size=50257)
```

These are **raw scores**, not probabilities yet -- one full 50,257-entry vocabulary distribution
(pre-softmax) per position.

**Step 4: Compute Loss (Cross-Entropy)**

For each real (non-padding) position -- the ones Step 2 did **not** mark `-100` -- we:

1. Convert that position's logits -> probabilities (softmax over all 50,257 vocab entries)
2. Look up the probability the model assigned to the **correct** next token (the label one position
   ahead, from Step 2's shift-by-one pairing)
3. Take the negative log of that probability -- this is the loss for that one position. A confident,
   *correct* guess (probability near 1) gives a loss near 0; a confident, *wrong* guess gives a very
   large loss, since `-log(x)` shoots toward infinity as `x` shrinks toward 0.
4. Average the per-position losses across every real token position -- the masked/padding positions
   from Step 2 are skipped entirely via `ignore_index=-100`, exactly the mechanism named there.

**Step 5: Backpropagation**

Compute gradients: `∂Loss/∂W` for every trainable parameter W in the model.

- **Full fine-tuning:** gradients flow to every parameter (~355M for `gpt2-medium`)
- **Partial freezing:** only the unfrozen last few blocks + head get real gradients
- **LoRA:** only the small adapter matrices get gradients (well under 1% of all parameters)

**Step 6: Optimizer Update**

Update weights in the direction that reduces loss:

```
W_new = W_old - learning_rate × gradient
```

**Step 7: Repeat**

Repeated across many steps, these tiny weight nudges accumulate into **learning**: the model becomes
better at predicting tokens that appear in our domain corpus.

---

**Key Takeaways:**

This is the first section that actually plots anything, so this is where we load the visualization
stack -- `matplotlib`/`seaborn` for the charts, `numpy` for the array math behind them. Every later
section that visualizes training internals reuses these same imports.

In [ ]:
# Visualization imports for intuition building
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle
from IPython.display import display, HTML
import warnings

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Visualization libraries loaded.")

### Running Steps 1-2 for Real

The walkthrough above described this in the abstract -- let's actually run it, one step at a time, on
`gpt2-medium` and our example paragraph. First, Steps 1-2: tokenize the sentence into fixed-length
tensors, then build `labels` (a clone of `input_ids`, with padding positions masked to `-100`) exactly
the way described above.

In [ ]:
import torch.nn.functional as F

example_text = (
    "Aria Voss stared at the signal counting itself out in prime numbers and felt the "
    "weight of two centuries press against her ribs."
)

# Step 1: tokenize
enc = tokenizer(
    example_text,
    truncation=True,
    max_length=128,
    padding="max_length",
    return_tensors="pt",
)
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)
real_len = int(attention_mask.sum().item())  # number of real (non-padding) tokens

# Step 2: build labels -- identical to input_ids, padding positions masked to -100
labels = input_ids.clone()
labels[attention_mask == 0] = -100  # mask padding, exactly like tokenize_causal()

print(
    f"Tokenized to {input_ids.shape[1]} total positions: {real_len} real tokens + "
    f"{input_ids.shape[1] - real_len} padding tokens"
)
print(f"first 8 input_ids : {input_ids[0, :8].tolist()}")
print(f"first 8 labels    : {labels[0, :8].tolist()}  <- identical to input_ids (not shifted)")
print(f"last 8 labels     : {labels[0, -8:].tolist()}  <- all -100 (padding, ignored by the loss)")


### Visual Guide: Masking and the Shift, Panel by Panel

Steps 1-2 above are dense in prose -- the figure right below turns them into a picture, built from the
*real* `input_ids`, `attention_mask`, and `labels` just computed for our example paragraph (nothing
here is a schematic with made-up numbers). Four panels, each isolating one piece of the mechanism:

- **Panel A** -- what `input_ids` actually holds: the real token stream, decoded back into text.
- **Panel B** -- what happens right at the real/padding boundary (position `real_len`): real tokens
  keep their own id as the label; padding positions get overwritten to `-100`.
- **Panel C** -- the shift itself: position `i`'s prediction is graded against the label sitting one
  slot ahead, `label[i+1]` -- the exact mechanism the write-up above described in words only.
- **Panel D** -- the payoff of `-100`: which positions the loss actually counts, and which it silently
  skips via `ignore_index=-100`.


In [ ]:

# Visual guide: masking and the shift, panel by panel -- built from the real tokenizer/model
# output above (input_ids, labels, attention_mask, real_len), not a fabricated schematic.
from matplotlib.patches import Patch


def _clean_tok(token_str):
    """Turn GPT-2 BPE's internal space/newline markers into something readable in a plot."""
    return token_str.replace("\u0120", "\u00b7").replace("\u010a", "\\n")


decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle(
    "Masking and Prediction, Panel by Panel -- real tokens from gpt2-medium's tokenizer",
    fontsize=13,
    fontweight="bold",
)

# Panel A: the real token stream -- what input_ids actually holds
ax_a = axes[0, 0]
n_show_a = 8
for pos in range(n_show_a):
    ax_a.add_patch(Rectangle((pos, 0), 0.9, 1, facecolor="lightblue", edgecolor="black"))
    ax_a.text(
        pos + 0.45, 0.5, _clean_tok(decoded_tokens[pos]),
        ha="center", va="center", fontsize=8,
    )
    ax_a.text(pos + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray")
ax_a.set_xlim(-0.2, n_show_a + 0.2)
ax_a.set_ylim(-0.6, 1.3)
ax_a.axis("off")
ax_a.set_title(
    "Panel A: input_ids -- the real token stream (position below each box)",
    fontsize=10, fontweight="bold",
)
ax_a.legend(
    handles=[Patch(facecolor="lightblue", edgecolor="black", label="Real token")],
    loc="lower center", bbox_to_anchor=(0.5, -0.18), fontsize=7,
)

# Panel B: the real/padding boundary -- where labels actually get masked to -100
ax_b = axes[0, 1]
start_b = max(0, real_len - 5)
end_b = min(input_ids.shape[1], real_len + 4)
window_b = list(range(start_b, end_b))
boundary_j = real_len - start_b
for j, pos in enumerate(window_b):
    is_real = pos < real_len
    color = "mediumseagreen" if is_real else "lightgray"
    label_text = _clean_tok(decoded_tokens[pos]) if is_real else "-100"
    ax_b.add_patch(Rectangle((j, 0), 0.9, 1, facecolor=color, edgecolor="black"))
    ax_b.text(j + 0.45, 0.5, label_text, ha="center", va="center", fontsize=8)
    ax_b.text(j + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray")
boundary_line_b = ax_b.axvline(
    boundary_j, color="red", linestyle="--", linewidth=1.5, label="Real/padding boundary"
)
ax_b.set_xlim(-0.2, len(window_b) + 0.2)
ax_b.set_ylim(-0.6, 1.3)
ax_b.axis("off")
ax_b.set_title(
    f"Panel B: labels right at the boundary (real_len={real_len})",
    fontsize=10, fontweight="bold",
)
ax_b.legend(
    handles=[
        Patch(facecolor="mediumseagreen", edgecolor="black", label="Real token (label = same token)"),
        Patch(facecolor="lightgray", edgecolor="black", label="Padding (label = -100)"),
        boundary_line_b,
    ],
    loc="lower center", bbox_to_anchor=(0.5, -0.3), fontsize=7,
)

# Panel C: the shift itself -- prediction at position i graded against label[i+1]
ax_c = axes[1, 0]
n_show_c = 6
for pos in range(n_show_c):
    ax_c.add_patch(Rectangle((pos, 0.7), 0.9, 1, facecolor="#ffd9a0", edgecolor="black"))
    ax_c.text(pos + 0.45, 1.2, _clean_tok(decoded_tokens[pos]), ha="center", va="center", fontsize=8)
    ax_c.text(pos + 0.45, 1.85, f"logits[{pos}]", ha="center", va="center", fontsize=6.5, color="gray")
    ax_c.add_patch(Rectangle((pos, -1.2), 0.9, 1, facecolor="lightblue", edgecolor="black"))
    ax_c.text(pos + 0.45, -0.7, _clean_tok(decoded_tokens[pos + 1]), ha="center", va="center", fontsize=8)
    ax_c.text(pos + 0.45, -1.45, f"label[{pos + 1}]", ha="center", va="center", fontsize=6.5, color="gray")
    ax_c.annotate(
        "", xy=(pos + 0.45, -0.15), xytext=(pos + 0.45, 0.65),
        arrowprops=dict(arrowstyle="->", color="darkred", lw=1.5),
    )
ax_c.set_xlim(-0.2, n_show_c + 0.2)
ax_c.set_ylim(-1.7, 2.2)
ax_c.axis("off")
ax_c.set_title(
    "Panel C: the shift -- position i's prediction is graded against label[i+1]",
    fontsize=10, fontweight="bold",
)
ax_c.legend(
    handles=[
        Patch(facecolor="#ffd9a0", edgecolor="black", label="Token at position i (what the model has read)"),
        Patch(facecolor="lightblue", edgecolor="black", label="label[i+1] -- what it's graded against"),
    ],
    loc="lower center", bbox_to_anchor=(0.5, -0.32), fontsize=7,
)

# Panel D: the payoff of -100 -- which positions the loss actually counts
ax_d = axes[1, 1]
for j, pos in enumerate(window_b):
    is_real = pos < real_len
    color = "coral" if is_real else "whitesmoke"
    ax_d.add_patch(Rectangle((j, 0), 0.9, 1, facecolor=color, edgecolor="black"))
    if not is_real:
        ax_d.text(
            j + 0.45, 0.5, "skipped", ha="center", va="center",
            fontsize=7, color="gray", fontweight="bold",
        )
    ax_d.text(j + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray")
boundary_line_d = ax_d.axvline(
    boundary_j, color="red", linestyle="--", linewidth=1.5, label="Real/padding boundary"
)
ax_d.set_xlim(-0.2, len(window_b) + 0.2)
ax_d.set_ylim(-0.6, 1.3)
ax_d.axis("off")
ax_d.set_title(
    "Panel D: ignore_index=-100 in action -- padding contributes zero loss",
    fontsize=10, fontweight="bold",
)
ax_d.legend(
    handles=[
        Patch(facecolor="coral", edgecolor="black", label="Counted in the loss"),
        Patch(facecolor="whitesmoke", edgecolor="black", label="Skipped (ignore_index=-100)"),
        boundary_line_d,
    ],
    loc="lower center", bbox_to_anchor=(0.5, -0.3), fontsize=7,
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"Real/padding boundary for our example paragraph: position {real_len} out of {input_ids.shape[1]}.")
print("Panels A-D all use the real tokenizer/model output computed above -- nothing here is fabricated.")


### Steps 3-5: Forward Pass, Loss, and Backprop in One Call

Passing `labels=...` into `base_model(...)` makes HuggingFace do Steps 3 and 4 internally in a single
call: it runs the forward pass (producing `outputs.logits`), then shifts and compares logits against
labels the way described above, returning the averaged result as `outputs.loss`. `step_loss.backward()`
is Step 5 -- PyTorch's autograd walks backward through every operation that produced `step_loss` and
computes `∂Loss/∂W` for every parameter that needs a gradient, without us deriving any calculus by hand.
`base_model.train()` beforehand just tells dropout-style layers to behave in "training mode" for this
one pass; it's switched back to `.eval()` a couple of cells down, once this illustrative pass is done,
so nothing here leaks into the rest of the notebook.

In [ ]:
base_model.train()  # need gradients for this one illustrative pass; restored to eval() below
base_model.zero_grad()
outputs = base_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
step_loss = outputs.loss
step_loss.backward()

print(f"logits shape: {tuple(outputs.logits.shape)}  (one 50,257-vocab prediction per position)")
print(f"average loss across all real positions: {step_loss.item():.3f}")


### Zooming Into the Loss: Per-Token Detail

`step_loss` above is already the *average* loss HuggingFace computed internally -- useful for training,
but it hides the position-by-position detail Step 4 actually describes. This cell recomputes that same
cross-entropy manually, one position at a time, using the exact `[:-1]` / `[1:]` alignment from
earlier, so we can see individual token losses instead of one averaged number.

In [ ]:
# Real per-position loss for the first few real (non-padding) tokens
shift_logits = outputs.logits[0, :-1, :]
shift_labels = labels[0, 1:]
per_token_loss = F.cross_entropy(
    shift_logits, shift_labels, reduction="none", ignore_index=-100
)
positions_to_show = min(8, real_len - 1)
losses_per_pos = per_token_loss[:positions_to_show].detach().cpu().numpy()

print(f"Per-token loss for the first {positions_to_show} real positions:")
print(losses_per_pos.round(3))
print(
    f"Mean of these {positions_to_show}: {losses_per_pos.mean():.3f}  "
    f"(compare to the full-sequence average loss printed above: {step_loss.item():.3f})"
)


### Measuring Real Gradient Flow Per Block

`step_loss.backward()` already populated `.grad` on every trainable parameter in `base_model`. This
cell just measures how large those gradients are, block by block, by taking the norm of every
parameter's gradient inside each of `gpt2-medium`'s 24 transformer blocks -- a direct, numeric answer
to "how much backprop signal reached this block."

In [ ]:
# Real gradient magnitude per transformer block (shows actual gradient flow, not a fabricated curve)
n_blocks = base_model.config.n_layer
block_grad_norms = []
for i in range(n_blocks):
    block_params = [
        p
        for n, p in base_model.named_parameters()
        if f"h.{i}." in n and p.grad is not None
    ]
    norm = (
        torch.norm(torch.stack([p.grad.norm() for p in block_params])).item()
        if block_params
        else 0.0
    )
    block_grad_norms.append(norm)

print(f"Computed a real gradient norm for all {n_blocks} transformer blocks.")
print(
    f"Block 0 (earliest): {block_grad_norms[0]:.4f}   "
    f"Block {n_blocks - 1} (latest): {block_grad_norms[-1]:.4f}"
)


### Step 6: The Actual Weight Update

This is the step every fine-tuning technique in this notebook is really about:
`W_new = W_old - learning_rate × gradient`. We're not letting an optimizer do this at scale yet (that
happens inside `Trainer.train()` in the very next section) -- instead, we manually apply that same
formula to one real weight from `base_model`, so the update is visible instead of buried inside
thousands of simultaneous parameter updates. One nudge is tiny: `lr=5e-5` times a small gradient often
works out to around `1e-8`, far too small to notice at 6 decimal places, which is why the printed delta
below uses scientific notation. `base_model.zero_grad()` and `.eval()` afterward reset the model back
to exactly how the rest of the notebook expects to find it -- this was a one-off illustration, not a
real training step, so nothing here is meant to persist. Once that's done, let's put all six steps
into one figure below.

In [ ]:
# Real weight update on one real parameter, using the actual computed gradient.
# LR=5e-5 times a small gradient is often ~1e-8 -- too small to see at 6 decimal places, so we
# print W_old/W_new AND the delta in scientific notation so the update is actually visible.
sample_name, sample_param = next(
    (n, p)
    for n, p in base_model.named_parameters()
    if p.grad is not None and p.dim() == 2
)
old_weight = sample_param.data.flatten()[0].item()
sample_grad = sample_param.grad.flatten()[0].item()
lr = 5e-5
weight_delta = -lr * sample_grad
new_weight = old_weight + weight_delta

base_model.zero_grad()
base_model.eval()  # leave base_model exactly as it was for the rest of the notebook

print(f"Parameter: {sample_name}")
print(f"W_old = {old_weight:.6f}   gradient = {sample_grad:.3e}   LR = {lr:.0e}")
print(f"\u0394W = -lr * gradient = {weight_delta:+.3e}   ->  W_new = {new_weight:.6f}")


In [ ]:
# Anatomy of one training step -- all six real numbers above, laid out in one figure
# (not fabricated numbers: every value below came from the cells above, computed for real on base_model)
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Anatomy of One Training Step (Continued Pretraining) -- real numbers from gpt2-medium",
    fontsize=13,
    fontweight="bold",
)

# Step 1: Tokenization
ax1 = axes[0, 0]
ax1.text(
    0.5,
    0.88,
    "Step 1: Tokenization",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax1.transAxes,
)
ax1.text(
    0.5,
    0.45,
    f'"{example_text[:38]}..."\n\u2193\n{input_ids[0, :8].tolist()} ...',
    ha="center",
    va="center",
    fontsize=9,
    transform=ax1.transAxes,
    bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
)
ax1.axis("off")
ax1.legend(
    handles=[Patch(facecolor="lightblue", alpha=0.7, edgecolor="black", label="Text -> token ids")],
    loc="lower center",
    fontsize=7,
)

# Step 2: The mask layout (real tokens vs. padding) -- the actual answer to "how is the mask laid out"
ax2 = axes[0, 1]
mask_row = attention_mask[0].cpu().numpy().reshape(1, -1)
ax2.imshow(
    mask_row, cmap="Greens", aspect="auto", vmin=0, vmax=1, extent=[0, 128, 0, 1]
)
boundary_line = ax2.axvline(
    real_len, color="red", linestyle="--", linewidth=1.5, label="Real/padding boundary"
)
ax2.set_yticks([])
ax2.set_xlabel("Token position (0-128)", fontsize=8)
ax2.set_title(
    f"Step 2: Mask Layout\n{real_len} real tokens (green, active) +\n"
    f"{128 - real_len} padding (white, labels=-100)",
    fontsize=9,
    fontweight="bold",
)
ax2.legend(
    handles=[
        Patch(facecolor="darkgreen", label="Real token (active)"),
        Patch(facecolor="white", edgecolor="black", label="Padding (labels=-100)"),
        boundary_line,
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=1,
    fontsize=7,
)

# Step 3: Forward pass -- real logits
ax3 = axes[0, 2]
real_logits = outputs.logits[0, :positions_to_show, :50].detach().cpu().numpy()
im3 = ax3.imshow(real_logits, cmap="RdYlBu_r", aspect="auto")
ax3.set_xlabel("Vocab (first 50 ids)", fontsize=8)
ax3.set_ylabel("Position", fontsize=8)
ax3.set_title(
    "Step 3: Forward Pass\nreal logits, first tokens", fontsize=9, fontweight="bold"
)
ax3.set_xticks([])
ax3.set_yticks([])
plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label="Logit value (raw score)")

# Step 4: Real per-position loss
ax4 = axes[1, 0]
positions = np.arange(len(losses_per_pos))
ax4.bar(
    positions, losses_per_pos, color="coral", alpha=0.8, width=0.6,
    label="Per-token loss",
)
ax4.axhline(
    losses_per_pos.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Mean (shown)",
)
ax4.set_xlabel("Token position", fontsize=9)
ax4.set_ylabel("Loss", fontsize=9)
ax4.set_title(
    f"Step 4: Compute Loss\nfull-sequence avg = {step_loss.item():.3f}",
    fontsize=9,
    fontweight="bold",
)
ax4.legend(fontsize=8)

# Step 5: Real gradient magnitude per block
ax5 = axes[1, 1]
tick_stride = max(1, n_blocks // 8)
ax5.barh(
    np.arange(n_blocks), block_grad_norms, color="purple", alpha=0.7,
    label="Gradient norm per block",
)
ax5.set_yticks(np.arange(0, n_blocks, tick_stride))
ax5.set_yticklabels([f"Block {i}" for i in range(0, n_blocks, tick_stride)], fontsize=8)
ax5.set_xlabel("Real gradient norm", fontsize=9)
ax5.set_title(
    "Step 5: Backpropagation\nactual per-block gradient norm",
    fontsize=9,
    fontweight="bold",
)
ax5.invert_yaxis()
ax5.legend(fontsize=7, loc="lower right")

# Step 6: Real weight update -- shown at enough precision to actually see the nudge
ax6 = axes[1, 2]
ax6.text(
    0.5,
    0.88,
    "Step 6: Update Weights",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.58,
    f"W_old = {old_weight:.6f}\ngradient = {sample_grad:.3e}\nLR = {lr:.0e}",
    ha="center",
    va="center",
    fontsize=10,
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.28,
    f"\u0394W = {weight_delta:+.3e}\nW_new = {new_weight:.6f}",
    ha="center",
    va="center",
    fontsize=11,
    transform=ax6.transAxes,
    color="green",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="lightgreen", alpha=0.5),
)
ax6.axis("off")
ax6.legend(
    handles=[Patch(facecolor="lightgreen", alpha=0.5, edgecolor="black", label="New (updated) weight")],
    loc="lower center",
    fontsize=7,
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"\n{'=' * 80}")
print(
    "Training Step Summary (all numbers above are real, from one forward+backward pass):"
)
print(f"{'=' * 80}")
print(f"1. Tokenize: {real_len} real tokens + {128 - real_len} padding tokens")
print("2. Mask layout: labels = input shifted left; padding positions set to -100")
print(f"3. Forward pass: logits shape {tuple(outputs.logits.shape)}")
print(f"4. Loss (avg over real tokens only): {step_loss.item():.3f}")
print(f"5. Backprop: gradients computed for all {n_blocks} transformer blocks")
print(
    f"6. Update: W_new = W_old + \u0394W, where \u0394W = -lr * gradient = {weight_delta:+.3e}"
)
print(
    "   That's why fine-tuning needs many steps: each one nudges a weight by a fraction of a "
    "percent, and Riverside's assistant only 'learns' after thousands of these tiny nudges add up."
)
print(f"{'=' * 80}")


---

## The Fine-Tuning Journey: A Problem-Solution Narrative

Rather than a flat taxonomy, fine-tuning is best understood as a **journey where each technique 
solves a problem left by the previous one**:

```mermaid
flowchart TD
    A[Pretrained Base Model] -->|Problem: Doesn't know your domain| B[Solution: Continued Pretraining]
    B -->|Problem: Continues text, won't follow instructions| C[Solution: Instruction Tuning SFT]
    C -->|Problem: Outputs aren't what humans prefer| D[Solution: Preference Alignment DPO/RLHF]
    
    style A fill:#e1f5ff
    style B fill:#b3e5fc
    style C fill:#81d4fa
    style D fill:#4fc3f7
```

At each stage, you can choose **how many parameters to update**:

| Approach | Trade-off | Use when |
|----------|-----------|----------|
| **Full fine-tuning** (100% params) | Max quality, max cost | Small models, abundant compute |
| **Partial freezing** (10-30% params) | Middle ground | Limited compute budget |
| **LoRA** (well under 1% params) | Min cost, swappable adapters | Most production scenarios |

**This notebook demonstrates:**
- All 3 data-based stages (continued pretraining, instruction tuning, preference alignment)
- All 3 parameter-based approaches (full, partial, LoRA)
- **Explained but not implemented in code:** PPO-based RLHF's optimization mechanics (clipped
  surrogate objective + KL penalty) are contrasted conceptually with DPO's approach in the DPO vs.
  PPO comparison further down (Concept 3) -- not a bare mention, but there's no runnable PPO training
  loop in this notebook.
- **Named but out of scope:** adapters, prefix tuning, QLoRA

---



## Concept 1 (Data-Based): Non-Instructional Fine-Tuning (Continued Pretraining)

**Riverside's question for this section:** does the model even know our characters and world exist
yet? Nothing downstream matters if it can't recognize "Aria Voss" or "the Meridian's Promise."

**What it is:** keep training with the exact same objective used for the original pretraining --
next-token prediction -- but on your own raw, unlabeled domain text instead of general web text. No
prompts, no "instructions", no labeled pairs: just plain paragraphs. This is often called _continued
pretraining_ or _domain-adaptive pretraining (DAPT)_.

**When to use it:** you have a pile of domain text (support tickets, legal filings, a publisher's back
catalog...) and you want the model to _absorb_ its vocabulary, facts, and style before you ever teach
it to follow instructions.

**Pros**

- Cheapest data to acquire -- no labeling/annotation needed, just clean text.
- Great at absorbing vocabulary, entities, and stylistic quirks (character names, invented
  terminology...).
- Simple training loop -- identical to pretraining (`labels = input_ids`).

**Cons**

- Does **not** teach the model to follow instructions or hold a conversation -- it only gets better
  at _continuing_ text like your domain text.
- Risk of shallow memorization instead of generalization if the corpus is small or repetitive.
- Risk of **catastrophic forgetting** of general-purpose ability if trained too long/aggressively.

Below we run this on a sample of chapters from Riverside's catalog, updating **all** of
`gpt2-medium`'s parameters (full fine-tuning -- more on that axis further down).


### Code Walkthrough: `tokenize_causal()` — Preparing Text for Next-Token Prediction

This is the first point in the notebook where we actually need to convert raw paragraph strings into
the fixed-length integer tensors a transformer consumes, so this is where `tokenize_causal()` gets
defined, right before the `dataset.map(...)` call that needs it. Every later stage that trains on
plain continuation text (partial freezing and LoRA continued pretraining, further down) reuses this
exact same function; the instruction-tuning and DPO sections swap in a response-masked variant
instead, since those need to hide the prompt from the loss.

```python
def tokenize_causal(examples, tokenizer, max_length=128):
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens
```

**Arguments:**

| Argument     | Type                  | Purpose                                                                                                                                            |
| ------------ | --------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------|
| `examples`   | `dict` (HF batch)     | A batch of examples with a `"text"` column — here, the raw paragraph strings in `non_inst_paragraphs`, wrapped in a `Dataset`.                     |
| `tokenizer`  | `PreTrainedTokenizer` | The tokenizer loaded in the baseline cell above (`AutoTokenizer.from_pretrained(MODEL_NAME)`). Passed in explicitly rather than closed over, so the same function works unchanged no matter which model/tokenizer this notebook is pointed at. |
| `max_length` | `int`, default `128`  | Hard cap on sequence length. Longer paragraphs are truncated; shorter ones are padded up to this length so every example in a batch has the same shape. |

**What it returns:** the usual tokenizer output (`input_ids`, `attention_mask`) plus a `labels` key,
which HuggingFace's `Trainer` requires to compute the causal-LM loss. `labels` starts as a copy of
`input_ids`, then every padding position (where `attention_mask == 0`) is overwritten with `-100` —
PyTorch's `CrossEntropyLoss` convention for "ignore this position." Without that mask, the model would
waste training signal learning to predict padding tokens instead of real text.

It's called in the cell below as `dataset.map(lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"])`.
`batched=True` is what makes `examples` a dict-of-lists (every text in the batch at once) instead of a
single example, which is why the list comprehension inside zips over `tokens["input_ids"]` rather than
indexing a single sequence.


In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

non_inst_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery"], max_chapters=3
)
print(
    f"Loaded {len(non_inst_paragraphs)} paragraphs from 3 novels for continued-pretraining demo"
)

non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})


def tokenize_causal(examples, tokenizer, max_length=128):
    """Standard next-token-prediction tokenization: labels = input_ids, with padding
    positions masked out (-100) so the loss ignores them."""
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


non_inst_tokenized = non_inst_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

Data's ready. Now load a **fresh, untouched copy** of `gpt2-medium` to actually fine-tune -- kept
separate from `base_model` so `base_model` stays the permanent "before" snapshot every later
comparison in this notebook relies on.

In [ ]:
full_ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
print(
    f"Loaded a fresh {MODEL_NAME} to fully fine-tune: "
    f"{sum(p.numel() for p in full_ft_model.parameters()):,} parameters, all trainable."
)

### Configuring and Running the Trainer

`TrainingArguments` + `Trainer` is HuggingFace's standard training loop -- it handles the batching,
the forward/backward pass, and the optimizer step described earlier in this notebook, so we don't
write that loop by hand. `max_steps=60` and `learning_rate=5e-5` keep this CPU demo fast; a real
Riverside training run would raise `max_steps` substantially. `trainer_full.train()` is the line that
actually runs all 60 of those steps -- this is the real training run every later comparison in this
notebook is measured against.

In [ ]:
training_args_full = TrainingArguments(
    output_dir="./checkpoints/non-instruction-full",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle; raise further for real runs
    logging_steps=10,
    save_strategy="no",
    learning_rate=5e-5,
    report_to="none",
)

trainer_full = Trainer(
    model=full_ft_model, args=training_args_full, train_dataset=non_inst_tokenized
)
trainer_full.train()
full_ft_model.save_pretrained("./checkpoints/non-instruction-full")
print("Saved continued-pretraining (full fine-tune) checkpoint.")

### Cracking Open Full Fine-Tuning: Which Blocks Actually Moved?

Full fine-tuning unfreezes every parameter, but that doesn't mean every block changes by the same
amount. Let's measure it directly: compare `full_ft_model`'s real weights, block by block, against
`base_model` -- the untouched pretrained checkpoint we've never trained. This is the same "crack it
open" treatment LoRA got earlier, applied to full fine-tuning.


In [ ]:
# Real per-block weight-delta norms: full_ft_model (after training) vs. base_model (never trained)
block_weight_deltas = []
base_state = dict(base_model.named_parameters())
for i in range(base_model.config.n_layer):
    delta_norm_sq = 0.0
    for name, p in full_ft_model.named_parameters():
        if f"h.{i}." in name:
            delta = p.data - base_state[name].data
            delta_norm_sq += delta.norm().item() ** 2
    block_weight_deltas.append(delta_norm_sq**0.5)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    range(len(block_weight_deltas)),
    block_weight_deltas,
    color="steelblue",
    alpha=0.85,
    label="||W_after - W_before|| (full fine-tuning)",
)
ax.set_xlabel("Transformer block")
ax.set_ylabel("||W_after - W_before|| (real)")
ax.set_title(
    "Full Fine-Tuning: Real Per-Block Weight Movement (full_ft_model vs. untouched base_model)",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="y")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(
    f"Every block moved (min \u0394 = {min(block_weight_deltas):.3f}, "
    f"max \u0394 = {max(block_weight_deltas):.3f}) -- full fine-tuning really does touch the whole "
    f"model, which is exactly why it's the most expensive option and the one most likely to nudge "
    f"Riverside's assistant away from plain English while it learns the catalog."
)

# Memory cleanup: full_ft_model's job is done -- non_instruct_ckpt (loaded from disk further down)
# takes over its role for every later comparison. On a laptop with a handful of gpt2-medium-sized
# models in memory at once, freeing this one now is the difference between "fits" and "swaps to disk."
import gc

del full_ft_model
gc.collect()
print(
    "Freed full_ft_model from memory (its checkpoint is saved to disk; non_instruct_ckpt reloads it later)."
)


### Visualizing Training Progress: Loss Curves

After training completes, let's look at what actually happened -- the real per-step loss recorded by
the `Trainer` above, not an idealized illustration. Textbook loss curves are smooth; a 60-step,
batch-size-2 CPU demo on a fresh model is usually much noisier, and that's worth seeing honestly.

**What to look for:**

1. **Downward trend:** Loss should decrease on average (model is learning), even if noisy step-to-step
2. **Convergence:** Loss should stop trending strongly downward by the end (not still falling fast)
3. **Magnitude:** Lower loss = better fit to domain text (but watch for overfitting on tiny corpora!)

**Reading a noisy real curve:** with only 6 logged points and batch_size=2, a single unusually easy or
hard paragraph can swing the reported loss by ±0.3 or more. Don't over-interpret small wiggles -- look
at the overall direction across all points, and compare against the other techniques' real curves
later in the notebook (instruction tuning, partial freezing, LoRA continued pretraining) to see which
setup is converging fastest for the same step budget.


In [ ]:
# Visualize the REAL loss curve from the continued-pretraining run above (trainer_full),
# not a fabricated "typical" curve -- this is exactly what your training just did.
def extract_loss_history(trainer):
    return [
        (entry["step"], entry["loss"])
        for entry in trainer.state.log_history
        if "loss" in entry
    ]


full_ft_history = extract_loss_history(trainer_full)
steps, losses = zip(*full_ft_history)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    steps,
    losses,
    marker="o",
    linewidth=2,
    markersize=7,
    color="green",
    label="Training loss",
)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title(
    f"Continued Pretraining (Full FT): real loss log ({losses[0]:.2f} \u2192 {losses[-1]:.2f})",
    fontsize=12,
    fontweight="bold",
)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\n{'=' * 70}")
print("Reading this REAL loss curve (not an idealized one):")
print(f"{'=' * 70}")
print(f"  Logged steps: {list(steps)}")
print(f"  Logged losses: {[round(l, 3) for l in losses]}")
print(f"  First -> last: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"{'=' * 70}")
print("What to look for:")
print(
    "  • A clean, monotonic plateau like a textbook figure is the exception, not the rule --"
)
print(
    "    especially at batch_size=2 with only a handful of steps, loss is dominated by"
)
print("    per-batch noise (which paragraph happened to be in this batch) more than by")
print("    the underlying trend.")
print(
    "  • If the trend is flat/noisy rather than decreasing: raise max_steps, increase the"
)
print(
    "    batch size, or train on more paragraphs so the trend has room to dominate the noise."
)
print(
    "  • Compare this to the loss curves for instruction tuning, partial freezing, and LoRA"
)
print(
    "    continued pretraining further down -- they were all recorded the same real way."
)
print(f"{'=' * 70}")

### Common Pitfalls: Continued Pretraining

**Pitfall #1: Catastrophic Forgetting**

**Bad:** Train for 10,000 steps on a tiny 50KB domain corpus  
**Good:** Train for 25-100 steps, then validate on general tasks (e.g., "The capital of France is...")

**Why it happens:** The model "overwrites" its general language knowledge with domain-specific patterns.

**How to avoid:**

- Keep training steps low initially (start with 25-50)
- Use a validation set with both domain AND general questions
- Watch for nonsense on general prompts (sign of forgetting)

---

**Pitfall #2: Shallow Memorization**

**Bad:** Tiny corpus (5KB), repeated 100 times → model memorizes exact phrases  
**Good:** Diverse corpus (500KB+) with varied writing styles

**How to detect:**

- Model completes prompts with **exact** training sentences (word-for-word)
- Model can't generalize to new prompts in the same style
- Perplexity drops close to its theoretical floor of **1.0** on training data (the model is nearly
  certain about every next token because it has seen this exact text before) but stays high on
  validation data it hasn't memorized

---

**Pitfall #3: Wrong Max Length**

**Bad:** `max_length=512` on a corpus of short sentences → 90% of every batch is padding  
**Good:** Match `max_length` to your typical paragraph length (128-256 for novels)

**Why it matters:** Wasted computation on padding, slower training, less effective learning

---

**Pitfall #4: No Tokenizer Padding Token**

**Bad:** Forget to set `tokenizer.pad_token` → crash or silent errors  
**Good:** Always set `tokenizer.pad_token = tokenizer.eos_token` for GPT-family models

---

**Quick Health Check After Training:**

```python
# Test 1: Domain knowledge (should work)
generate(model, "Aria Voss checked the Meridian's Promise and")

# Test 2: General knowledge (should still work!)
generate(model, "The capital of France is")

# Test 3: Novel generalization (should work, not memorize)
generate(model, "In the Under-Hold, the rebels gathered and")
```

If test 2 fails → you overtrained (catastrophic forgetting).  
If test 3 is word-for-word from training → shallow memorization.


## Concept 2 (Data-Based): Instructional (Supervised) Fine-Tuning

**Riverside's question for this section:** the model now knows the lore -- but can a ghostwriter
_ask_ it for a continuation, or does it just ramble? An assistant nobody can direct isn't an
assistant.

### The Problem with Continued Pretraining Alone

After continued pretraining, the model knows your domain vocabulary and can continue text in your
style. But try asking it a question:

**You:** `"What are the five tides in the Tidebound world?"`  
**Model (after continued pretraining):** `"What are the five tides in the Tidebound world? This 
question has puzzled scholars for centuries. Some believe there are actually six tides, while..."`
(continues rambling)

**The problem:** The model learned to _continue_ prose, not to _answer questions_ or _follow
instructions_. It will keep generating narrative-style text forever because that's what it was trained
on.

### The Solution: Instruction Tuning (Supervised Fine-Tuning / SFT)

**What it is:** Train on `(prompt, completion)` pairs where the **prompt** is an instruction/question
and the **completion** is the desired response. Crucially, we **mask the prompt tokens** in the loss
so the model is only penalized for the completion portion.

**Key insight:** This teaches the model _behavior_ -- "when you see input shaped like X, respond like
Y" -- rather than just "keep talking like this corpus."

Real-world instruction datasets include:

- [alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned) - 52K instruction-following examples
- [OpenOrca](https://huggingface.co/datasets/Open-Orca/OpenOrca) - 4.2M GPT-4 completions
- [OpenAssistant/oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1) - 161K human-rated conversations

Here we auto-derive a tiny instruction dataset from Riverside's catalog:

- **Prompt:** `"Continue the fiction narrative in the same style: <paragraph N>"`
- **Completion:** `<paragraph N+1>`

**Pros:**

- Model learns to _follow a format/instruction_, not just continue prose
- Directly usable for chat/assistant interfaces
- Loss masking means the model isn't penalized for "predicting" the prompt it didn't generate

**Cons:**

- Needs actual (prompt, completion) pairs (expensive to create by hand)
- Can narrow diversity toward the exact template it was trained on
- Doesn't fix preference issues (model might follow instructions but in an unhelpful way)

This cell introduces **LoRA** (parameter-efficient tuning) to keep training fast on CPU -- foreshadowing
the budget conversation Riverside's IT lead is going to have with us later.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

INSTRUCTION_PREFIX = "Continue the fiction narrative in the same style:\n\n"


def build_instruction_pairs(novels=None, max_chapters=3):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "cyberpunk", "literary"]  # 5-genre default for richer style diversity

    pairs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        for path in sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]:
            paras = [
                p.strip().replace("\n", " ")
                for p in path.read_text(encoding="utf-8").split("\n\n")
                if len(p.strip()) > 200
            ]
            for a, b in zip(paras, paras[1:]):
                pairs.append(
                    {"prompt": f"{INSTRUCTION_PREFIX}{a}\n\n", "completion": b}
                )
    return pairs

### Tokenizing With the Prompt-Mask Pattern

`build_instruction_pairs()` gave us `(prompt, completion)` strings; `tokenize_instruction()` converts
one pair into the actual tensors `Trainer` needs, using the same prompt-masking idea introduced
earlier in this notebook: `labels` starts as a full copy of the tokenized text, then every **prompt**
position (not just padding) gets set to `-100`, so the loss only ever grades the completion.

In [ ]:
def tokenize_instruction(example, max_length=160, prompt_max_length=96):
    prompt_ids = tokenizer(
        example["prompt"], truncation=True, max_length=prompt_max_length
    )["input_ids"]
    full_text = example["prompt"] + example["completion"]
    tokens = tokenizer(
        full_text, truncation=True, padding="max_length", max_length=max_length
    )
    labels = tokens["input_ids"].copy()
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100  # don't compute loss on the prompt portion
    for i, mask in enumerate(tokens["attention_mask"]):
        if mask == 0:
            labels[i] = -100  # don't compute loss on padding either
    tokens["labels"] = labels
    return tokens

### Building the Instruction Dataset

Now actually build the pairs from 5 genres and tokenize every one of them with the function above.

In [ ]:
instruction_pairs = build_instruction_pairs(
    novels=["scifi", "fantasy", "mystery", "cyberpunk", "literary"], max_chapters=2
)
print(f"Built {len(instruction_pairs)} instruction pairs from 5 novels")

instruction_dataset = Dataset.from_list(instruction_pairs)
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction, remove_columns=["prompt", "completion"]
)

### Wrapping the Base Model in a LoRA Adapter

This is the first LoRA usage in the notebook -- see the dedicated LoRA section further down for the
full math; for now, `get_peft_model()` freezes every weight in a fresh base model and injects small
trainable adapter matrices into each `c_attn` projection, so only those tiny matrices get optimizer
state during training.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT-2's combined attention projection
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = get_peft_model(instruct_base, lora_config)
instruct_lora_model.print_trainable_parameters()

### Training and Saving the Adapter

Same `Trainer` pattern as continued pretraining, just with the LoRA-wrapped model, the prompt-masked
dataset, and a higher learning rate (`2e-4` vs. `5e-5`) -- LoRA needs a higher LR since it's only
updating a tiny slice of parameters.

In [ ]:
training_args_instruct = TrainingArguments(
    output_dir="./checkpoints/instruction-lora",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained("./checkpoints/instruction-lora")
print("Saved instruction-tuned LoRA adapter.")

### Code Walkthrough: Instruction Tuning, Recapped

**What just ran, across the last several cells -- four conceptual steps, each already introduced
briefly right before its own code. This cell ties them together with the details that don't fit in
a one-paragraph intro:**

---

**Step A: `build_instruction_pairs()` — creating (prompt, completion) pairs**

```python
pairs.append({"prompt": f"{INSTRUCTION_PREFIX}{a}\n\n", "completion": b})
```

For every pair of consecutive paragraphs `(a, b)` in each chapter, the _preceding_ paragraph becomes the prompt and the _next_ paragraph becomes the completion. This is the cheapest way to auto-generate instruction pairs from raw prose — no human labelling required. The `"\n\n"` delimiter marks where the model should stop echoing the prompt and start generating.

---

**Step B: `tokenize_instruction()` — the prompt-mask pattern**

This is the core difference from continued pretraining's `tokenize_causal()`:

```
Continued pretraining:  [-100 for padding only,  real labels for everything else]
Instruction tuning:     [-100 for prompt + pad,  real labels for completion only]
```

In code:

```python
for i in range(min(len(prompt_ids), len(labels))):
    labels[i] = -100   # mask the entire prompt portion
```

Setting `labels[i] = -100` at prompt positions tells `F.cross_entropy(ignore_index=-100)` to skip those positions when computing the loss. The model is only graded on the _completion_ tokens — never penalised for "predicting" the instruction it was given.

---

**Step C: `LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"])` — LoRA hyperparameters**

| Param                       | Value          | What it controls                                             |
| --------------------------- | -------------- | ------------------------------------------------------------ |
| `r=8`                       | Rank           | Bottleneck dimension — 8 basis vectors to express the update |
| `lora_alpha=16`             | Scaling        | Effective LR multiplier = `alpha/r` = 2.0                    |
| `target_modules=["c_attn"]` | Which layers   | GPT-2 packs Q, K, V into one `c_attn` projection             |
| `lora_dropout=0.05`         | Regularization | Randomly zeros adapter activations during training           |

`get_peft_model(base, config)` wraps every targeted layer with a `LoraLayer` object and freezes all other weights — the only parameters that get optimizer state are the two small adapter matrices per layer.

---

**Step D: `Trainer.train()` — what HuggingFace's Trainer does for you**

Under the hood, one `Trainer.train()` call:

1. Iterates over `train_dataset` in mini-batches of `per_device_train_batch_size=2`
2. Calls `model.forward(input_ids, attention_mask, labels)` → computes cross-entropy loss (ignoring `labels=-100` positions)
3. Calls `loss.backward()` → computes gradients only for `requires_grad=True` parameters (the LoRA matrices)
4. Calls `optimizer.step()` → updates those parameters by `lr × gradient`
5. Logs the loss every `logging_steps=10` steps

6. Stops after `max_steps=60` regardless of dataset size

### The Instruction-Tuning Mask Layout, For Real

Contrast this with the continued-pretraining mask layout earlier in the notebook: there, only
**padding** was masked, and every real token was active. Here, the **prompt itself is masked too** --
the model is only ever penalized for generating the completion, never for reproducing the prompt it
was given. The cell below takes one real `(prompt, completion)` pair from `instruction_pairs`, runs it
through the real `tokenize_instruction()` used for training, and colors every token position by what
the label mask actually does with it.


In [ ]:
# Real mask layout for instruction tuning: prompt masked (-100), completion active, padding masked
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

example_pair = instruction_pairs[0]
encoded_example = tokenize_instruction(example_pair)
example_labels = np.array(encoded_example["labels"])
example_attention = np.array(encoded_example["attention_mask"])

# Classify every position: 0 = masked prompt, 1 = active completion, 2 = masked padding
region = np.zeros(len(example_labels), dtype=int)
region[example_attention == 0] = 2  # padding
region[(example_attention == 1) & (example_labels != -100)] = 1  # completion (active)
# everything else (attention==1 & labels==-100) is the masked prompt, stays 0

prompt_masked = int(np.sum(region == 0))
completion_active = int(np.sum(region == 1))
padding_masked = int(np.sum(region == 2))

fig, ax = plt.subplots(figsize=(14, 2.2))
cmap = ListedColormap(["lightgray", "mediumseagreen", "white"])
ax.imshow(
    region.reshape(1, -1),
    cmap=cmap,
    aspect="auto",
    vmin=0,
    vmax=2,
    extent=[0, len(region), 0, 1],
)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title(
    f"{prompt_masked} prompt tokens masked + {completion_active} completion tokens active "
    f"+ {padding_masked} padding tokens masked",
    fontsize=10,
    fontweight="bold",
)
legend_handles = [
    Patch(
        facecolor="lightgray", edgecolor="black", label="Prompt (masked, labels=-100)"
    ),
    Patch(
        facecolor="mediumseagreen",
        edgecolor="black",
        label="Completion (active, real labels)",
    ),
    Patch(facecolor="white", edgecolor="black", label="Padding (masked, labels=-100)"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.55),
    ncol=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(f"Prompt text:     {example_pair['prompt'][:80]!r}...")
print(f"Completion text: {example_pair['completion'][:80]!r}...")
print(
    f"\nMasked (prompt): {prompt_masked} tokens | Active (completion): {completion_active} tokens "
    f"| Masked (padding): {padding_masked} tokens"
)
print(
    "Compare to continued pretraining: there, every real token was active. Here, the prompt is "
    "masked too, so the model only ever gets gradient signal from the completion."
)

### Common Pitfalls: Instruction Tuning

**Pitfall #1: Forgetting to Mask the Prompt**

**Bad:** Compute loss on both prompt AND completion → model "predicts" the prompt it was given  
**Good:** Set prompt tokens to `-100` in labels → only penalized for the completion

**Why it matters:**

Without masking:

```python
labels = [3, 822, 25, ...]  # entire sequence including prompt
```

With masking:

```python
labels = [-100, -100, -100, 822, 25, ...]  # first 10 tokens (prompt) masked
```

The model should only learn to **generate the completion**, not memorize the prompt.

---

**Pitfall #2: Template Over-Fitting**

**Bad:** All training pairs use identical prefix: `"Continue the narrative: ..."`  
**Good:** Vary the instruction format, or accept this if you'll always use that prefix at inference

**What happens:** Model becomes "allergic" to prompts without the exact prefix. If you prompt with
just the raw paragraph (no prefix), it won't know what to do.

**Fix:** Either:

1. Use diverse instruction templates during training
2. Always use the exact same prefix at inference (consistency is key)

---

**Pitfall #3: Completion Too Short/Long**

**Bad:** Completions are 5 tokens on average → model learns to be terse  
**Good:** Completions should match your inference expectation (50-100 tokens for narrative)

**Why:** The model learns the **distribution of lengths** from training. If all completions are short,
it will always generate short responses, even when you want more detail.

---

**Pitfall #4: Wrong Learning Rate**

**Bad:** Use the same LR as pretraining (5e-5) for LoRA  
**Good:** LoRA needs **higher LR** (2e-4 to 5e-4) because you're only updating a tiny subset of params

**Rule of thumb:**

- Full fine-tuning: 5e-5 to 1e-4
- Partial freezing: 1e-4 to 2e-4
- LoRA: 2e-4 to 5e-4

Lower rank → higher LR (more aggressive updates needed).

---

**Quick Health Check After Instruction Tuning:**

```python
# Test 1: With the instruction prefix (should work)
prompt = INSTRUCTION_PREFIX + "Aria checked the Meridian and\\n\\n"
generate(instruct_model, prompt)

# Test 2: Without prefix (will likely fail if over-fitted to template)
generate(instruct_model, "Aria checked the Meridian and")

# Test 3: Novel instruction (should generalize)
prompt = INSTRUCTION_PREFIX + "In the Upper decks, Marcus\\n\\n"
generate(instruct_model, prompt)
```

If test 2 produces nonsense → template over-fitting (model expects the prefix).  
If test 3 produces off-topic output → not enough diverse training data.


In [ ]:
# Quick health check after instruction tuning: actually run the three tests described above
print(
    "=== Test 1: With the instruction prefix (should follow the fiction-continuation format) ==="
)
prompt_with_prefix = INSTRUCTION_PREFIX + "Aria checked the Meridian and\n\n"
print(generate(instruct_lora_model, prompt_with_prefix), "\n")

print("=== Test 2: Without the prefix (checks for template over-fitting) ===")
print(generate(instruct_lora_model, "Aria checked the Meridian and"), "\n")

print("=== Test 3: Novel instruction (checks generalization to an unseen prompt) ===")
prompt_novel = INSTRUCTION_PREFIX + "In the Upper decks, Marcus\n\n"
print(generate(instruct_lora_model, prompt_novel))

## Concept 3 (Data-Based): Preference Alignment (RLHF / DPO)

**Riverside's question for this section:** the model follows instructions now -- but does it write
the way Riverside's editors _actually_ like, or just "a" technically-valid continuation? An assistant
that's correct-but-unusable still gets ignored.

### The Problem with Instruction Tuning Alone

After instruction tuning, the model follows instructions. But it might produce outputs that are
_technically correct_ but not what humans actually want:

**An editor asks:** `"Explain quantum entanglement."`  
**Instruction-tuned model:** `"Quantum entanglement is a phenomenon where particles become correlated 
such that the quantum state of one particle cannot be described independently... [continues for 10 
paragraphs with excessive jargon]"`

**Problems:**

- Too verbose (they wanted a 2-sentence explanation)
- Wrong tone (too academic for a casual question)
- Doesn't prioritize what the reader cares about

**The core insight:** Instruction tuning teaches the model to _respond_, but not which responses
humans _prefer_.

### The Solution: Preference Alignment

**The idea:** Show the model pairs of responses to the same prompt -- one that humans prefer
(`chosen`) and one they don't (`rejected`) -- and train it to increase the probability of preferred
responses.

**Two approaches:**

1. **RLHF (Reinforcement Learning from Human Feedback):** Train a separate reward model to score
   responses, then use PPO (reinforcement learning) to optimize the LLM against that reward. (Complex,
   not demoed here.)
2. **DPO (Direct Preference Optimization):** Skip the reward model entirely and optimize directly
   against preference pairs with a closed-form loss. (Simpler, demoed below.)

Real-world preference datasets:

- [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) - 170K human preferences on
  helpfulness/harmlessness
- [ultrafeedback-binarized-preferences-cleaned](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned)
  - 60K preference pairs

**Our stand-in for "what Riverside's editors prefer":**

- **Chosen:** The paragraph that actually follows in the novel (coherent, on-topic continuation --
  the closest thing we have to "an editor approved this")
- **Rejected:** An unrelated paragraph from a different chapter (off-topic, worse continuation)

This is a stand-in for real human preference labels, but demonstrates the mechanics. In a real
Riverside deployment, `chosen`/`rejected` pairs would come from editors actually rating two draft
continuations against each other -- worth keeping in mind for the "how much data would we really need"
question the training run below raises.


### DPO Intuition: What the Loss Is Actually Rewarding

Forget the arithmetic for a second -- here's the whole idea in one sentence: **DPO nudges the model to
like the `chosen` response a bit more, and the `rejected` response a bit less, than a frozen copy of
itself did.**

Walking through it as a story, not a computation:

1. We have two responses to the same prompt: one we like (`chosen`), one we don't (`rejected`).
2. The **policy model** (the one being trained) has some current opinion of how likely each response
   is. So does the **frozen reference model** (a snapshot from before DPO started).
3. For each response, DPO asks: _"Did the policy move its opinion up or down, compared to the
   reference?"_ That movement is the **margin**.
4. DPO then compares the two margins: _"Did the policy move up on `chosen` by more than it moved up
   on `rejected`?"_ If yes, good -- the model is learning the right preference. If no, the loss will
   push back.
5. That comparison gets squashed through a sigmoid (so it behaves like a probability) and turned into
   a loss with a `-log`, exactly like ordinary binary classification: "which response wins?"

The reference model matters because it's an **anchor**: without it, nothing would stop the policy from
just assigning every response a probability of 1 and calling it a day (mode collapse). By measuring
_relative_ movement instead of absolute probability, DPO can only reward genuine preference for
`chosen` over `rejected`, not "be confident about everything."

**Pros:** no separate reward model (unlike PPO-based RLHF), works well with LoRA, directly optimizes
for human preference.

**Cons:** needs paired preference data, can over-optimize ("reward hacking") if beta is too high or
data is noisy, needs a frozen reference model in memory.

<details>
<summary><strong>Optional: the closed-form loss</strong> (skip this if the five steps above are enough -- nothing below adds a new idea, it just names the pieces mathematically)</summary>

$$
\mathcal{L}_{DPO} = -\log \sigma\Big(\beta\big[(\log \pi_\theta(y_w \mid x) - \log \pi_{ref}(y_w
\mid x)) - (\log \pi_\theta(y_l \mid x) - \log \pi_{ref}(y_l \mid x))\big]\Big)
$$

$\pi_\theta$ = current policy, $\pi_{ref}$ = frozen reference, $y_w$ = chosen, $y_l$ = rejected,
$\sigma$ = sigmoid, $\beta$ = temperature. This is exactly steps 3-5 above written symbolically, and
it's the same loss `trl.DPOTrainer` implements -- useful if you're reading the paper or production
code side by side with this notebook, not required to understand what the training loop below does.

</details>

The training loop below is a **simplified, from-scratch implementation** so you can see the mechanics;
it also records the real margin/loss at every step so the chart right after it reflects what actually
happened during training, not a toy example. Production code would use `trl.DPOTrainer`.


In [ ]:
import copy
import torch.nn.functional as F


def build_preference_pairs(novels=None, max_chapters=4, max_pairs=30):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "horror", "literary"]  # 5-genre default

    chapter_files = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        chapter_files.extend(sorted(novel_path.glob("chapter-*.txt"))[:max_chapters])

    all_paragraphs = []
    for path in chapter_files:
        paras = [
            p.strip().replace("\n", " ")
            for p in path.read_text(encoding="utf-8").split("\n\n")
            if len(p.strip()) > 200
        ]
        all_paragraphs.append(paras)

    pairs = []
    for c_idx, paras in enumerate(all_paragraphs):
        other_chapter = all_paragraphs[(c_idx + 1) % len(all_paragraphs)]
        for i in range(len(paras) - 1):
            prompt = f"{INSTRUCTION_PREFIX}{paras[i]}\n\n"
            chosen = paras[i + 1]  # the real, on-topic continuation
            rejected = other_chapter[
                i % len(other_chapter)
            ]  # an unrelated paragraph -> a worse continuation
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pairs[:max_pairs]

### Encoding a (Prompt, Response) Pair for Logprob Math

DPO needs to score whole responses, not just next tokens, so `encode_response()` tokenizes
`prompt + response` together and builds a `response_mask` -- `1` only over the response portion --
the same masking idea as instruction tuning's prompt mask, just used here to select what
`sequence_logprob()` (defined a couple cells down) sums over instead of what the loss ignores.

In [ ]:
def encode_response(prompt, response, max_length=160, prompt_max_length=96):
    """Tokenize prompt+response and return a response_mask marking only the response
    tokens (excluding the shared prompt and any padding) as targets for logprob math."""
    prompt_ids = tokenizer(prompt, truncation=True, max_length=prompt_max_length)[
        "input_ids"
    ]
    full = tokenizer(
        prompt + response, truncation=True, padding="max_length", max_length=max_length
    )
    response_mask = [0] * max_length
    start = min(len(prompt_ids), max_length)
    end = min(sum(full["attention_mask"]), max_length)
    for i in range(start, end):
        response_mask[i] = 1
    return {
        "input_ids": torch.tensor(full["input_ids"]).unsqueeze(0).to(device),
        "attention_mask": torch.tensor(full["attention_mask"]).unsqueeze(0).to(device),
        "response_mask": torch.tensor(response_mask).unsqueeze(0).to(device),
    }

### Scoring a Full Response: `sequence_logprob()`

This is the piece DPO's math actually needs from each model: given a `(prompt, response)` pair, how
likely did the model think that whole response was? `log_softmax` + `gather` picks out the log-prob
of the actual next token at every position, and multiplying by `response_mask` before summing means
only the response's tokens count -- exactly the quantity `log π(y | x)` in the DPO formula from the
markdown before the training loop below.

In [ ]:
def sequence_logprob(model, input_ids, attention_mask, response_mask):
    """Sum of log P(token_t | tokens<t) over the response-mask positions only."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    targets = input_ids[:, 1:]
    mask = response_mask[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_logprobs = torch.gather(log_probs, 2, targets.unsqueeze(-1)).squeeze(-1)
    return (token_logprobs * mask).sum(dim=-1)

### Setting Up the Policy, Frozen Reference, and Optimizer

`policy_model` is the model DPO actually trains -- it's the same `instruct_lora_model` from the
instruction-tuning section, continuing where that left off. `reference_model` is a frozen
`copy.deepcopy` snapshot of it at this exact moment -- the anchor the earlier DPO-intuition markdown
described, needed so margins measure genuine drift instead of both models moving together. Only the
LoRA adapter's parameters go into the optimizer, same as instruction tuning.

In [ ]:
BETA = 0.1
policy_model = instruct_lora_model  # continue tuning the instruction-tuned LoRA adapter
reference_model = copy.deepcopy(policy_model).eval()
for p in reference_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    [p for p in policy_model.parameters() if p.requires_grad], lr=1e-5
)

### Building the Preference Pairs

`build_preference_pairs()` (defined at the top of this section) generates `(prompt, chosen, rejected)`
triples from the corpus -- `chosen` is the real next paragraph, `rejected` is an unrelated one from a
different chapter. `dpo_history` is created empty here so the training loop below can record real
per-step numbers for the chart right after it, instead of an illustrative curve.

In [ ]:
preference_pairs = build_preference_pairs(
    novels=["scifi", "fantasy", "mystery", "horror", "literary"], max_chapters=4, max_pairs=30
)
print(f"Built {len(preference_pairs)} preference pairs from 5 novels for the DPO demo")

# Recorded so the visualization right after this cell plots what ACTUALLY happened here,
# instead of a fabricated/illustrative curve.
dpo_history = {
    "step": [],
    "loss": [],
    "chosen_margin": [],
    "rejected_margin": [],
    "preference_diff": [],
}

### The DPO Training Loop

Everything above was setup; this is where it all comes together, one preference pair at a time:
encode `chosen` and `rejected`, score both under the policy and the frozen reference via
`sequence_logprob()`, compute the two margins and the DPO loss from the formula in the earlier
markdown, then `loss.backward()` + `optimizer.step()` -- the same backprop/update mechanics from
"Anatomy of One Training Step" near the top of this notebook, just written out by hand instead of
inside `Trainer.train()`, because DPO's loss needs both models' outputs at once.

In [ ]:
policy_model.train()
for step, pair in enumerate(preference_pairs):
    chosen = encode_response(pair["prompt"], pair["chosen"])
    rejected = encode_response(pair["prompt"], pair["rejected"])

    policy_chosen_lp = sequence_logprob(policy_model, **chosen)
    policy_rejected_lp = sequence_logprob(policy_model, **rejected)
    with torch.no_grad():
        ref_chosen_lp = sequence_logprob(reference_model, **chosen)
        ref_rejected_lp = sequence_logprob(reference_model, **rejected)

    chosen_margin = policy_chosen_lp - ref_chosen_lp
    rejected_margin = policy_rejected_lp - ref_rejected_lp
    logits = BETA * (chosen_margin - rejected_margin)
    loss = -F.logsigmoid(logits).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    dpo_history["step"].append(step)
    dpo_history["loss"].append(loss.item())
    dpo_history["chosen_margin"].append(chosen_margin.mean().item())
    dpo_history["rejected_margin"].append(rejected_margin.mean().item())
    dpo_history["preference_diff"].append(
        (chosen_margin - rejected_margin).mean().item()
    )

    if step % 5 == 0:
        print(f"step {step:02d} | dpo_loss={loss.item():.4f}")

policy_model.save_pretrained("./checkpoints/preference-dpo")
print("Saved DPO-aligned adapter.")

### Code Walkthrough: The DPO Training Loop, Recapped

**What just ran, across the last several cells -- the core DPO math implemented in plain PyTorch,
introduced piece by piece above. This cell connects those pieces and fills in the details that don't
fit in a one-paragraph intro:**

---

**Step A: `encode_response()` — response mask**

```python
response_mask[i] = 1   # only mark completion tokens as targets
```

Unlike the `Trainer`-based cells, the DPO loop computes log-probabilities manually. `encode_response` tokenizes `prompt + response` together, then builds a `response_mask` that is `1` only over the completion portion. This allows `sequence_logprob` to sum `log P(token | context)` only over the _response_ — not the prompt, not the padding.

---

**Step B: `sequence_logprob()` — how to score a full response**

```python
log_probs = F.log_softmax(logits, dim=-1)           # (batch, seq_len, vocab)
token_logprobs = torch.gather(log_probs, 2, targets.unsqueeze(-1)).squeeze(-1)  # pick the actual next-token's log prob
return (token_logprobs * mask).sum(dim=-1)          # sum over response positions only
```

- `F.log_softmax(logits, dim=-1)` → log-probabilities over the full 50K vocab
- `torch.gather(..., targets)` → extracts only the log-prob of the _ground-truth_ next token at each position
- `* mask` → zeros out prompt positions; `.sum()` → total sequence log-likelihood

---

**Step C: The DPO loss formula**

```python
chosen_margin   = policy_chosen_lp  - ref_chosen_lp    # policy drift for preferred response
rejected_margin = policy_rejected_lp - ref_rejected_lp  # policy drift for dispreferred response
logits_dpo = BETA * (chosen_margin - rejected_margin)
loss = -F.logsigmoid(logits_dpo).mean()
```

- `chosen_margin - rejected_margin > 0` → the policy has shifted _more_ toward chosen than rejected (desired)
- `BETA` controls how strongly the preference margin is enforced (typical: 0.1–0.3)
- `-log sigmoid(...)` is the DPO objective from [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290) — binary cross-entropy over a preference logit rather than token-level loss

---

**Step D: Why `reference_model` must be frozen**

```python
reference_model = copy.deepcopy(policy_model).eval()
for p in reference_model.parameters():
    p.requires_grad = False
```

DPO measures how far the _policy_ has drifted from a stable anchor. If the reference also changed during training, the margins `(policy_lp - ref_lp)` would be meaningless — both models could drift in the same direction and the loss would never converge. `copy.deepcopy` makes a full independent snapshot; setting `requires_grad=False` ensures the reference accumulates _no_ gradients and its weights never move.

---

**Step E: Optimizer scope — only LoRA matrices get updated**

```python
optimizer = torch.optim.AdamW(
    [p for p in policy_model.parameters() if p.requires_grad], lr=1e-5
)
```


Because `policy_model` is the instruction-tuned LoRA adapter, only the adapter matrices have `requires_grad=True`. The frozen GPT-2 base and the reference model never accumulate gradients — optimizer state stays tiny, just like during instruction tuning.

In [ ]:
# Visualize the DPO training dynamics we just recorded above -- real numbers, not a toy example,
# so this chart actually reflects the training that happened in the previous cell.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
steps = np.array(dpo_history["step"])

axes[0].plot(
    steps,
    dpo_history["chosen_margin"],
    "g-o",
    label="chosen margin vs. ref",
    linewidth=2,
    markersize=4,
)
axes[0].plot(
    steps,
    dpo_history["rejected_margin"],
    "r-o",
    label="rejected margin vs. ref",
    linewidth=2,
    markersize=4,
)
axes[0].axhline(0, color="black", linestyle="--", alpha=0.4)
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("Log-prob margin vs. reference")
axes[0].set_title("Policy Drift From Reference")
axes[0].legend(fontsize=8, loc="best")
axes[0].grid(alpha=0.3)

preference_diff = np.array(dpo_history["preference_diff"])
axes[1].plot(steps, preference_diff, "b-o", linewidth=2, markersize=4)
axes[1].axhline(0, color="black", linestyle="--", alpha=0.4, label="No preference")
axes[1].fill_between(
    steps,
    0,
    preference_diff,
    where=preference_diff > 0,
    alpha=0.3,
    color="green",
    label="Prefers chosen",
)
axes[1].fill_between(
    steps,
    0,
    preference_diff,
    where=preference_diff < 0,
    alpha=0.3,
    color="red",
    label="Prefers rejected",
)
axes[1].set_xlabel("Training Step")
axes[1].set_ylabel("Preference difference")
axes[1].set_title("Preference Margin Over Training")
axes[1].legend(fontsize=8, loc="best")
axes[1].grid(alpha=0.3)

axes[2].plot(steps, dpo_history["loss"], "m-o", linewidth=2, markersize=4)
axes[2].set_xlabel("Training Step")
axes[2].set_ylabel("DPO Loss")
axes[2].set_title(
    f"Loss: {dpo_history['loss'][0]:.3f} \u2192 {dpo_history['loss'][-1]:.3f}"
)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("DPO Training Summary (real numbers from the run above):")
print(f"  Initial preference margin: {dpo_history['preference_diff'][0]:.3f}")
print(f"  Final preference margin:   {dpo_history['preference_diff'][-1]:.3f}")
print(f"  Loss: {dpo_history['loss'][0]:.4f} -> {dpo_history['loss'][-1]:.4f}")

# Honest reaction to whatever actually happened above -- not an assumed success story.
# Riverside's stakes: this ran on 30 synthetic preference pairs. One afternoon of two editors
# rating draft continuations against each other wouldn't produce many more than that in real life.
margin_improved = dpo_history["preference_diff"][-1] > dpo_history["preference_diff"][0]
loss_improved = dpo_history["loss"][-1] < dpo_history["loss"][0]
print(f"\n{'=' * 80}")
if margin_improved and loss_improved:
    print(
        "Reading this run: preference margin AND loss both moved in the right direction -- "
        "on this run's numbers, 30 pairs was (barely) enough signal."
    )
else:
    print(
        "Reading this run HONESTLY: the numbers above did not cleanly improve "
        f"({'margin regressed' if not margin_improved else 'margin OK'}, "
        f"{'loss went up' if not loss_improved else 'loss OK'}). That's a real, useful result for "
        "Riverside, not a notebook bug: with only 30 preference pairs and one pass through them, "
        "there isn't enough signal for DPO to reliably converge. Before shipping preference "
        "alignment to production, Riverside would need either (a) meaningfully more preference "
        "pairs (hundreds to thousands, not 30), (b) multiple epochs over the pairs it has, or "
        "(c) a smaller learning rate with more steps -- not just 'run DPO and ship it.'"
    )
print(f"{'=' * 80}")

### DPO vs. PPO: Two Ways to Optimize the Same Preference Data

The intuition and formula above are DPO's. Before moving on to common pitfalls, it's worth answering
the question the "Two approaches" list at the top of this section raised but didn't unpack: **what
does PPO-based RLHF actually do differently, and why did this notebook pick DPO?**

**PPO Intuition, Story First**

Forget the equations for a second -- here's PPO's whole idea in one sentence, the same way the DPO
intuition above did it: **PPO also wants the policy to prefer `chosen` over `rejected`, but instead
of reading that preference off a fixed pair, it has the policy try a response, grades it, and only
trusts each grade a little at a time.**

Walking through it as a story, not a computation:

1. Somebody has to grade a response. That's the **reward model** -- trained separately, ahead of
   time, on human preference pairs like the ones DPO uses -- so it can hand back a single number for
   *any* `(prompt, response)`, not just the ones it was trained on.
2. The **policy** being trained produces a response and the reward model scores it. A **frozen
   reference policy** -- an older, stable snapshot -- also has an opinion on how likely that exact
   response was. This is the same "policy vs. frozen anchor" setup DPO already uses above.
3. PPO asks the same question DPO's margin asks: *"compared to the reference, did the policy just
   get more or less likely to produce this response?"* PPO measures that as a raw probability
   **ratio** (new likelihood ÷ old likelihood) rather than DPO's log-probability difference, but it's
   the same underlying idea: how far did the policy move.
4. Multiply that ratio by the reward score. A response that scored well *and* that the policy moved
   toward gets reinforced; a response that scored poorly but the policy moved toward anyway gets
   pushed back down.
5. Here's the one genuinely new idea DPO doesn't need: **clip the ratio** before multiplying it by
   the reward. If a single update would make the policy wildly more (or less) likely to produce a
   response, PPO caps how much of that swing counts, so one noisy reward-model score can't send the
   policy off a cliff in one step. A KL penalty adds a second, gentler leash on top, continuously
   pulling the policy back toward the reference.

DPO can skip both the reward model (step 1) and the clip/leash (step 5) because it always trains on
the *same fixed* `chosen`/`rejected` pair -- there's no live, possibly-noisy reward score to guard
against, and no self-generated response whose distribution could suddenly shift underneath the
training loop. That's the trade PPO makes: it can optimize *any* reward signal, not just static
pairs, at the cost of needing a reward model and two different stability mechanisms (clip + KL) to
keep that added flexibility from blowing up.

**The same story, in the standard RLHF vocabulary:** PPO-based RLHF (the original InstructGPT/ChatGPT
recipe) is a two-stage, on-policy pipeline -- train a reward model on preference pairs, then have the
policy generate its own responses, score them with that reward model, and update the policy with the
clipped surrogate objective plus KL penalty from the story above. **DPO skips the reward model
entirely.** [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290) show the same RLHF objective has
a closed-form solution directly in terms of the policy's own log-probabilities on the *existing*
preference pairs -- no reward model, no on-policy generation, no clipping, no separate value network.
That's the entire reason the training loop earlier in this section could be a handful of PyTorch
lines.

|                          | **PPO-based RLHF**                                                                           | **DPO** (used above)                                                        |
| ------------------------ | --------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------ |
| Needs a trained reward model? | Yes -- a separate model, its own training pass                                            | No -- the loss reads the preference pairs directly                             |
| Data used per update     | Responses the policy *generates itself*, scored live (on-policy)                                | The same static `(chosen, rejected)` pairs, reused every epoch                 |
| Optimization objective   | Clipped surrogate objective + KL penalty vs. reference                                          | One closed-form cross-entropy-style loss                                       |
| Extra moving parts       | Reward model + value/baseline network + rollout sampling                                        | Just the frozen reference model this section already built                     |
| Stability in practice    | Sensitive to reward-model quality, clip range, KL coefficient -- notoriously fiddly to tune      | Comparatively stable, fewer knobs to turn                                      |
| Compute cost             | Highest of every technique in this notebook (reward model + generation + PPO updates)            | About the same cost as instruction tuning                                      |
| Still worth it when...   | The reward signal can't be reduced to static pairs (live user feedback, unit-test pass/fail, a rule-based checker) | Preference data is already collected as pairs -- exactly Riverside's situation |

A full from-scratch implementation would reuse the exact same `encode_response()` / `sequence_logprob()`
helpers and preference pairs as the DPO loop above -- but this notebook stops at the conceptual
comparison: production PPO's learned reward model and on-policy rollout sampling are a bigger build
than this notebook takes on, and the table above already isolates the one thing that's genuinely
different between the two methods on identical data -- the optimization mechanics, not the reward
signal or the dataset.


### Common Pitfalls: DPO (Preference Alignment)

**Pitfall #1: Weak or Noisy Preference Pairs**

**Bad:** `chosen` and `rejected` are nearly identical or have inconsistent quality  
**Good:** Clear preference signal — chosen should be noticeably better than rejected

**Example of BAD pair:**

- Chosen: `"Quantum entanglement means particles are connected."`
- Rejected: `"Quantum entanglement means particles are linked."` ← too similar!

**Example of GOOD pair:**

- Chosen: `"Quantum entanglement means particles are connected."`
- Rejected: `"This is a complicated quantum mechanical phenomenon involving wave function collapse and non-local correlations that [500 more words of jargon]"` ← clearly worse!

**Why it matters:** If the preference signal is weak, the model won't learn what humans actually want.

---

**Pitfall #2: Beta (β) Tuned Incorrectly**

**Bad:** β = 1.0 → over-aggressive preference signal, mode collapse  
**Bad:** β = 0.01 → too weak, model barely changes  
**Good:** Start with β = 0.1 to 0.3 and tune based on validation

**What β does:**

- **High β (0.5-1.0):** Strong preference signal → fast learning, but risk of "reward hacking" (model
  exploits the preference distribution)
- **Low β (0.01-0.05):** Weak signal → slow learning, safer
- **Medium β (0.1-0.3):** Balanced (most common in practice)

**Symptom of bad β:**

- β too high → model generates identical text for all prompts (mode collapse)
- β too low → model ignores preferences entirely, no improvement

---

**Pitfall #3: Forgetting to Freeze the Reference Model**

**Bad:** Reference model continues training → DPO loss becomes meaningless  
**Good:** `reference_model.eval()` and `param.requires_grad = False` for all ref params

**Why:** The reference model is the "anchor" — it must stay fixed to measure how far the policy has
drifted. If it changes, the loss calculation breaks.

---

**Pitfall #4: Training on Top of Base Model Instead of Instruction-Tuned**

**Bad:** Run DPO on a raw pretrained model  
**Good:** DPO should be the **final stage** after instruction tuning

**Why:** DPO assumes the model already knows how to follow instructions. If you DPO a base model, it
will learn preferences over gibberish continuations, not helpful responses.

**Correct pipeline:**

```
Base model → Continued pretraining → Instruction tuning → DPO
```

---

**Pitfall #5: Not Monitoring Divergence from Reference**

**Bad:** Train DPO for 1000 steps without checking KL divergence  
**Good:** Monitor `KL(policy || reference)` — stop if it exceeds 1.0-2.0

**Why:** DPO can cause the policy to drift too far from the reference, leading to nonsensical but
"high-scoring" outputs (reward hacking).

---

**Quick Health Check After DPO:**

```python
# Test 1: Preferred style (should be concise)
generate(dpo_model, INSTRUCTION_PREFIX + "Explain quantum entanglement.\\n\\n")

# Test 2: Still coherent on domain tasks
generate(dpo_model, INSTRUCTION_PREFIX + "Continue: Aria checked the panel and\\n\\n")

# Test 3: Doesn't mode collapse (vary prompts, should get varied outputs)
for i in range(3):
    generate(dpo_model, INSTRUCTION_PREFIX + f"Describe the Meridian (attempt {i}).\\n\\n")
```

If test 1 is still verbose → β too low or not enough training.  
If test 3 produces identical output 3 times → mode collapse (β too high).


In [ ]:
# Quick health check after DPO: actually run the three tests described above
print(
    "=== Test 1: Preferred style (should be more concise than the pre-DPO instruction model) ==="
)
print(
    generate(policy_model, INSTRUCTION_PREFIX + "Explain quantum entanglement.\n\n"),
    "\n",
)

print("=== Test 2: Still coherent on domain tasks ===")
print(
    generate(
        policy_model, INSTRUCTION_PREFIX + "Continue: Aria checked the panel and\n\n"
    ),
    "\n",
)

print("=== Test 3: Doesn't mode collapse (vary prompts, should get varied outputs) ===")
for i in range(3):
    print(f"--- attempt {i} ---")
    print(
        generate(
            policy_model,
            INSTRUCTION_PREFIX + f"Describe the Meridian (attempt {i}).\n\n",
        )
    )

---

## End of Part 1: What's Been Trained So Far

Three checkpoints exist on disk now, all saved under `./checkpoints/` (relative to the workspace
root, not this notebook's folder -- see the `CONTENT_DIR` resolution cell near the top for why):

| Checkpoint on disk                 | What it is                                              |
| ----------------------------------- | -------------------------------------------------------- |
| `./checkpoints/non-instruction-full` | Continued pretraining, full fine-tuning (Concept 1)      |
| `./checkpoints/instruction-lora`     | Instruction tuning, LoRA adapter (Concept 2)              |
| `./checkpoints/preference-dpo`       | Preference alignment, DPO on top of the instruction adapter (Concept 3) |

Riverside now has a model that knows its catalog, follows instructions, and has attempted (with mixed
results, honestly reported above) to match editor preference. What's still unanswered: **how many of
the model's weights did each of those stages actually need to update, and is there a cheaper way?**
That's the parameter-based axis -- continue to
**[Part 2: Parameter-Based Techniques + QLoRA & Quantization](02-llm-finetuning-parameter-techniques.ipynb)**,
which reloads these three checkpoints from disk and trains three more (full fine-tuning parameter
count, partial freezing, and LoRA continued pretraining) before introducing QLoRA and a real,
runnable look at post-training quantization.
